## Multi-Agent RAN Slicing Environment

This module defines a custom **multi-agent reinforcement learning environment**
for simulating 5G Radio Access Network (RAN) slicing, built on top of
[PettingZoo's](https://pettingzoo.farama.org/) `ParallelEnv` API. It models how
independent RL agents compete and cooperate for shared radio resources
(PRBs and transmit power) across different 5G service types.

### Network Slices Modeled

| Slice | Full Name | Priority | Key SLA |
|-------|-----------|----------|---------|
| **eMBB** | Enhanced Mobile Broadband | Throughput | ≥ 50 Mbps, ≤ 500 ms latency |
| **URLLC** | Ultra-Reliable Low-Latency Communication | Latency & Reliability | ≤ 250 ms latency, ≤ 1e-4 packet loss |
| **mMTC** *(optional)* | massive Machine-Type Communication | Device density | ≥ 95% connection density, ≤ 1000 ms latency |

Each slice is controlled by its own agent (`embb_agent`, `urllc_agent`, and
optionally `mmtc_agent`), which outputs a 2D continuous action
`[PRB_fraction, tx_power_level]` at every timestep.

### Core Modeling Features

- **3GPP TR 38.901-compliant channel model** — Urban Macro (UMa) NLOS pathloss,
  correlated log-normal shadowing, and independent Rayleigh small-scale fading
  per UE, with optional random-walk UE mobility.
- **Multi-UE support per slice** — traffic demand is fixed at the *slice* level
  (not per UE), so PRBs/power are split across UEs. This means adding more UEs
  increases contention and degrades per-UE service — the physically correct
  behavior — rather than artificially scaling total demand.
- **Fluid-queue dynamics** — each UE has its own queue, updated each step as
  `Q[t] = clip(Q[t-1] + arrivals - service, 0, Q_max)`. Latency is derived via
  Little's Law (`W = Q / service_rate`) plus fixed transmission/processing delays.
  Buffer overflow produces packet loss.
- **Time-varying traffic** — diurnal (24-hour cyclic) load patterns for eMBB and
  URLLC, each randomized per episode (phase, base level, amplitude) plus bounded
  noise; mMTC traffic follows periodic reporting bursts.
- **Reward design** — each agent gets a blended reward: a slice-specific term
  (e.g., URLLC uses a multiplicative "AND" combination of mean latency, tail
  latency, and reliability rewards) plus a shared cooperative global term, so
  agents are partially incentivized to consider system-wide performance.
- **Constraint costs** — a separate, log-scaled cost signal per slice (0 = SLA met,
  up to 1 = severe violation), useful for constrained/safe RL formulations
  (e.g., Lagrangian methods, CMDPs) independent of the reward.

### API Surface

Implements the standard PettingZoo parallel API:
- `reset(seed=None, options=None) → (observations, infos)`
- `step(actions) → (observations, rewards, terminations, truncations, infos)`
- `observation_space(agent)`, `action_space(agent)`
- `state()` — flattened global state for centralized-critic training (e.g., MAPPO)
- `render()` — human-readable per-step debug printout

### Configuration

All environment behavior is controlled via `__init__` arguments — no hardcoded
paths or environment-specific values. Key options:

| Parameter | Purpose |
|-----------|---------|
| `max_steps` | Episode length (timesteps) |
| `total_prbs`, `max_power_w`, `bandwidth_per_prb` | Radio resource budget |
| `enable_mmtc` | Toggle the optional mMTC agent/slice |
| `time_varying_traffic`, `enable_mobility` | Toggle dynamic vs. static conditions |
| `embb_max_queue`, `urllc_max_queue` | Per-slice buffer size (Mbits) |
| `num_ues` | Dict of UEs per slice, e.g. `{"embb": 3, "urllc": 2}` |
| `seed` | RNG seed for reproducibility |

In [ ]:
import numpy as np
from gymnasium import spaces
from pettingzoo import ParallelEnv
from typing import Dict, List, Tuple, Optional
import math
from collections import deque


class Constants:
    """Physical layer and model constants"""
    EPS = 1e-8
    TIMESTEP_MS = 1.0
    TIMESTEP_SEC = TIMESTEP_MS / 1000.0
    EMBB_BASE_RATE_MBPS = 50.0
    URLLC_BASE_RATE_MBPS = 10.0
    MMTC_BASE_RATE = 50.0
    MMTC_PERIODIC_AMPLITUDE = 0.3
    MMTC_PERIODIC_CYCLE = 10
    # NEW: 3GPP TR 38.901 Parameters
    CARRIER_FREQ_GHZ = 2.0   # fc
    UE_HEIGHT_M = 1.5        # h_UT
    SHADOWING_STD = 6.0      # dB (from standard for UMa NLOS)
    URLLC_PROCESSING_DELAY_MS = 0.5
    URLLC_TRANSMISSION_DELAY_MS = 0.5
    EMBB_TRANSMISSION_DELAY_MS = 1.0


class ImprovedRANSlicingEnv(ParallelEnv):
    """
    Enhanced RAN Slicing Environment with realistic radio dynamics,
    multiple slice types, and time-varying conditions.

    Multi-UE model
    --------------
    * The total traffic arriving at a slice is FIXED regardless of how many
      UEs share it (traffic_intensity x BASE_RATE_MBPS at the slice level).
    * PRBs and power are divided equally among the UEs in each slice.
    * Each UE therefore sees a smaller service rate -> deeper queues and higher
      latency as UE count grows, which is the physically correct behaviour.
    * Every UE carries its own independent small-scale fading component on top
      of the shared slice-level pathloss + shadowing.
    * Slice-level metrics are aggregated from per-UE results:
        throughput    -> sum
        latency       -> mean  (tail captured via history in reward)
        packet_loss   -> mean
        queue_length  -> sum
        spectral_eff  -> mean
    """
    metadata = {"name": "ran_slicing_v2", "render_modes": ["human"]}

    def __init__(
        self,
        max_steps: int = 200,
        total_prbs: int = 100,
        max_power_w: float = 20.0,
        bandwidth_per_prb: float = 180e3,   # 180 kHz
        enable_mmtc: bool = False,
        time_varying_traffic: bool = True,
        enable_mobility: bool = True,
        embb_max_queue: float = 500.0,      # Mbits  (total slice buffer)
        urllc_max_queue: float = 500.0,       # Mbits  (total slice buffer)
        # NEW: number of UEs per slice. Defaults to 1 UE each (original behaviour).
        # Example: {"embb": 3, "urllc": 2, "mmtc": 5}
        num_ues: Optional[Dict[str, int]] = None,
        seed: Optional[int] = 4,
    ):
        super().__init__()

        # ------------------------------------------------------------------
        # Core configuration
        # ------------------------------------------------------------------
        self.max_steps = max_steps
        self.total_prbs = total_prbs
        self.max_power_w = max_power_w
        self.bandwidth_per_prb = bandwidth_per_prb
        self.enable_mmtc = enable_mmtc
        self.time_varying_traffic = time_varying_traffic
        self.enable_mobility = enable_mobility
        self.embb_max_queue = embb_max_queue
        self.urllc_max_queue = urllc_max_queue

        # ------------------------------------------------------------------
        # UE counts per slice (default = 1 → identical to original)
        # ------------------------------------------------------------------
        _default_ues: Dict[str, int] = {"embb": 1, "urllc": 1, "mmtc": 1}
        if num_ues is not None:
            _default_ues.update(num_ues)
        self.num_ues: Dict[str, int] = {
            k: max(1, int(v)) for k, v in _default_ues.items()
        }

        self.step_count = 0
        self._rng = np.random.RandomState(seed)

        # ------------------------------------------------------------------
        # Agents
        # ------------------------------------------------------------------
        self.agents = ["embb_agent", "urllc_agent"]
        if enable_mmtc:
            self.agents.append("mmtc_agent")
        self.possible_agents = self.agents[:]

        self.agent_to_slice = {"embb_agent": "embb", "urllc_agent": "urllc"}
        if enable_mmtc:
            self.agent_to_slice["mmtc_agent"] = "mmtc"

        # ------------------------------------------------------------------
        # QoS Constraints (5G-aligned)
        # ------------------------------------------------------------------
        self.constraints = {
            "embb":  {"min_throughput": 50.0, "max_latency": 500.0}, # Mbps, ms
            "urllc": {"min_throughput": 10.0, "max_latency": 250,   # Mbps, ms - Changed
                      "max_packet_loss": 1e-4},                     # 99.99% Reliability
            "mmtc":  {"max_latency": 1000.0,  "min_connection_density": 0.95, # ms
                      "max_energy_per_bit": 100.0},                 # nJ/bit
        }

        # ------------------------------------------------------------------
        # Observation space bounds (ordered, semantic, safe)
        # Layout:
        # [curr(5), hist(5), channel(2), traffic(1), time_sin, time_cos]
        # ------------------------------------------------------------------
        def box(low, high):
            return spaces.Box(np.array(low, dtype=np.float32),
                              np.array(high, dtype=np.float32), dtype=np.float32)

        self._observation_spaces = {
            "embb_agent": box(
                [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  -5, -10,  0,  -1, -1],
                [200, 500, 1000, 500, 15,  200, 500, 1000, 500, 15,  30, 10,  5,  1, 1],
            ),
            "urllc_agent": box(
                [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  -5, -10,  0,  -1, -1],
                [50, 1000, 1000, 5, 15,  50, 1000, 1000, 5, 15,  30, 10,  1,  1, 1],
            ),
        }
        if enable_mmtc:
            self._observation_spaces["mmtc_agent"] = box(
                [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  -5, -10,  0,  -1, -1],
                [100, 1000, 100, 200, 50,  100, 1000, 100, 200, 50,  30, 10,  50,  1, 1],
            )

        # ------------------------------------------------------------------
        # Action space
        # [PRB_fraction, tx_power_level] ∈ [0, 1]
        # ------------------------------------------------------------------
        self._action_spaces = {
            agent: spaces.Box(low=0.0, high=1.0, shape=(2,), dtype=np.float32)
            for agent in self.agents
        }

        # ------------------------------------------------------------------
        # Physical tracking (enforced in step)
        # ------------------------------------------------------------------
        self.actual_prb_allocation = {"embb": 0.0, "urllc": 0.0}
        if enable_mmtc:
            self.actual_prb_allocation["mmtc"] = 0.0

        self.constants = Constants()

        # ------------------------------------------------------------------
        # Initialize environment state
        # ------------------------------------------------------------------
        self._reset_state()

    # ------------------------------------------------------------------
    # State initialisation
    # ------------------------------------------------------------------

    def _reset_state(self):
        """Initialise all environment state variables."""

        # ------------------------------------------------------------
        # Slice metrics (only create enabled slices)
        # ------------------------------------------------------------
        self.metrics = {
            "embb":  {"throughput": 0.0, "latency": 5.0,  "packet_loss": 0.0,
                      "queue_length": 0.0, "spectral_efficiency": 0.0},
            "urllc": {"throughput": 0.0, "latency": 0.5,  "packet_loss": 0.0,
                      "queue_length": 0.0, "spectral_efficiency": 0.0},
        }
        if self.enable_mmtc:
            self.metrics["mmtc"] = {
                "throughput": 0.0, "latency": 0.0, "connection_density": 0.0,
                "energy_efficiency": 0.0, "active_devices": 0.0,
            }

        # ------------------------------------------------------------
        # Metric history (fixed-size, warm-started)
        # ------------------------------------------------------------
        self.metric_history = {}
        for sl, m in self.metrics.items():
            self.metric_history[sl] = {}
            for key, val in m.items():
                dq = deque(maxlen=50)
                dq.append(val)
                self.metric_history[sl][key] = dq

        # ------------------------------------------------------------
        # Per-UE queue state (one queue per UE per slice)
        # Queues are tracked internally; slice-level metrics are aggregates.
        # Each UE owns buffer = total_max / n_ues.
        # ------------------------------------------------------------
        self.ue_queue_lengths: Dict[str, List[float]] = {
            "embb":  [0.0] * self.num_ues["embb"],
            "urllc": [0.0] * self.num_ues["urllc"],
        }
        if self.enable_mmtc:
            self.ue_queue_lengths["mmtc"] = [0.0] * self.num_ues["mmtc"]

        # ------------------------------------------------------------
        # Channel conditions (SINR in dB)
        # Slice-level channel (pathloss + shadowing, shared by all UEs in the
        # slice). Per-UE independent Rayleigh fades are drawn in the helper.
        # ------------------------------------------------------------
        CHANNEL_STD = 2.0
        self.channel_quality = {}
        self.shadowing = {}
        for sl in self.metrics:
            self.shadowing[sl] = -abs(self._rng.randn() * CHANNEL_STD) # always <= 0
            self.channel_quality[sl] = -100.0

        self.distance: Dict[str, float] = {}

        # ------------------------------------------------------------
        # Traffic parameters (episode-level randomness)
        # ------------------------------------------------------------
        self.traffic_params = {
            "embb": {
                "phase": self._rng.uniform(0, 24),
                # Base multiplier 1.2 = 60 Mbps average load
                "base": self._rng.uniform(1.1, 1.3),
                # Amplitude 0.8 = +40 Mbps at peak, -40 Mbps at trough
                "amplitude": self._rng.uniform(0.7, 0.9),
            },
            "urllc": {
                "phase": self._rng.uniform(0, 24),
                # Base multiplier 1.5 = 15 Mbps average load
                "base": self._rng.uniform(1.4, 1.6),
                # Amplitude 1.0 = +10 Mbps at peak, -10 Mbps at trough
                "amplitude": self._rng.uniform(0.9, 1.1),
            },
        }

        self.traffic_intensity = {
            "embb":  self.traffic_params["embb"]["base"],
            "urllc": self.traffic_params["urllc"]["base"],
        }

        if self.enable_mmtc:
            self.traffic_intensity["mmtc"] = self.constants.MMTC_BASE_RATE

        # ------------------------------------------------------------
        # Resource allocation tracking
        # Action dim = 2 → [prb_fraction, tx_power]
        # ------------------------------------------------------------
        self.previous_allocation = {
            agent: np.zeros(2, dtype=np.float32) for agent in self.agents
        }
        self.actual_prb_allocation   = {"embb": 0, "urllc": 0}
        self.actual_power_allocation = {"embb": 0.0, "urllc": 0.0}
        if self.enable_mmtc:
            self.actual_prb_allocation["mmtc"]   = 0
            self.actual_power_allocation["mmtc"] = 0.0

        # ------------------------------------------------------------
        # Energy & episode statistics
        # ------------------------------------------------------------
        self.total_energy = 0.0
        self.episode_stats = {
            "total_embb_violations":  0,
            "total_urllc_violations": 0,
            "mean_embb_throughput":   [],
            "mean_urllc_latency":     [],
        }

    # ------------------------------------------------------------------
    # PettingZoo API
    # ------------------------------------------------------------------

    def observation_space(self, agent):
        return self._observation_spaces[agent]

    def action_space(self, agent):
        return self._action_spaces[agent]

    def reset(self, seed=None, options=None):
        """Reset environment to initial state"""
        if seed is not None:
            self._rng = np.random.RandomState(seed)
        self.step_count = 0
        self.agents = self.possible_agents[:]
        self._reset_state()
        return {a: self._get_observation(a) for a in self.agents}, {a: {} for a in self.agents}

    def step(self, actions):
        """Execute one environment step"""
        # ------------------------------------------------------------
        # Advance time & Parse actions
        # ------------------------------------------------------------
        self.step_count += 1
        allocations = self._parse_actions(actions)

        # ------------------------------------------------------------
        # Update exogenous dynamics
        # ------------------------------------------------------------
        if self.time_varying_traffic:
            self._update_traffic_patterns()
        if self.enable_mobility:
            self._update_channel_conditions()

        # ------------------------------------------------------------
        # Resource allocation
        # ------------------------------------------------------------
        prb_allocation = self._allocate_prbs(allocations)
        self.actual_prb_allocation = prb_allocation.copy() # expose for logging

        power_allocation = self._allocate_power(allocations)
        self.actual_power_allocation = power_allocation.copy() # expose for logging

        power_levels = {
            "embb_agent":  power_allocation["embb"],
            "urllc_agent": power_allocation["urllc"],
        }
        if self.enable_mmtc:
            power_levels["mmtc_agent"] = power_allocation["mmtc"]

        # ------------------------------------------------------------
        # Slice updates
        # ------------------------------------------------------------
        self._update_embb_slice(prb_allocation["embb"],  power_levels["embb_agent"])
        self._update_urllc_slice(prb_allocation["urllc"], power_levels["urllc_agent"])
        if self.enable_mmtc:
            self._update_mmtc_slice(prb_allocation["mmtc"], power_levels["mmtc_agent"])

        # ------------------------------------------------------------
        # Rewards & costs
        # ------------------------------------------------------------
        rewards = self._compute_rewards(allocations)
        costs   = self._compute_costs()

        # ------------------------------------------------------------
        # Episode statistics (constraint tracking)
        # ------------------------------------------------------------
        if costs["embb"]  > 0.0: self.episode_stats["total_embb_violations"]  += 1
        if costs["urllc"] > 0.0: self.episode_stats["total_urllc_violations"] += 1
        self.episode_stats["mean_embb_throughput"].append(self.metrics["embb"]["throughput"])
        self.episode_stats["mean_urllc_latency"].append(self.metrics["urllc"]["latency"])

        # ------------------------------------------------------------
        # Termination logic & return compilation
        # ------------------------------------------------------------
        done = self.step_count >= self.max_steps
        terminations = {a: done for a in self.agents}
        truncations  = {a: False for a in self.agents}
        observations = {a: self._get_observation(a) for a in self.agents}
        infos        = self._build_infos(costs)

        if done:
            # Attach episode summary (PettingZoo style)
            for a in self.agents:
                infos[a]["episode"] = {
                    "embb_violation_rate":  self.episode_stats["total_embb_violations"]  / self.max_steps,
                    "urllc_violation_rate": self.episode_stats["total_urllc_violations"] / self.max_steps,
                    "mean_embb_throughput": float(np.mean(self.episode_stats["mean_embb_throughput"])),
                    "mean_urllc_latency":   float(np.mean(self.episode_stats["mean_urllc_latency"])),
                }
            # Clear agents list (PettingZoo requirement)
            self.agents = []

        return observations, rewards, terminations, truncations, infos

    # ------------------------------------------------------------------
    # Action parsing
    # ------------------------------------------------------------------

    def _parse_actions(self, actions):
        """Parse and clip actions to valid ranges [0.0, 1.0]"""
        return {agent: np.clip(action, 0.0, 1.0) for agent, action in actions.items()}

    # ------------------------------------------------------------------
    # Resource allocation (unchanged from original)
    # ------------------------------------------------------------------

    def _allocate_prbs(self, allocations):
        """
        Allocate PRBs using minimum guarantees + proportional sharing.
        allocations: agent -> action vector [prb_fraction, tx_power]
        Returns: slice -> integer PRBs
        """
        # Minimum PRB guarantees (protect URLLC)
        min_prb_fraction = {"embb": 0.01, "urllc": 0.01}
        if self.enable_mmtc:
            min_prb_fraction["mmtc"] = 0.05

        min_prbs  = {sl: max(1, int(f * self.total_prbs)) for sl, f in min_prb_fraction.items()}
        guaranteed = sum(min_prbs.values())
        available  = self.total_prbs - guaranteed

        # Edge case: guarantees exceed capacity
        if available <= 0:
            scale = self.total_prbs / guaranteed
            return {k: max(1, int(v * scale)) for k, v in min_prbs.items()}

        # Extract PRB requests from agent actions
        requests = {"embb": allocations["embb_agent"][0], "urllc": allocations["urllc_agent"][0]}
        if self.enable_mmtc:
            requests["mmtc"] = allocations["mmtc_agent"][0]

        # Normalize requests
        total_req = sum(requests.values())
        requests  = ({k: v / total_req for k, v in requests.items()} if total_req > self.constants.EPS
                     else {k: 1.0 / len(requests) for k in requests})

        # Proportional allocation of remaining PRBs
        desired    = {k: v * available for k, v in requests.items()}
        additional = {k: int(v) for k, v in desired.items()}
        fractional = {k: desired[k] - additional[k] for k in desired}
        remaining  = available - sum(additional.values())

        # Handle rounding leftovers
        if remaining > 0:
            for sl, _ in sorted(fractional.items(), key=lambda x: x[1], reverse=True)[:remaining]:
                additional[sl] += 1

        # Final allocation
        alloc = {sl: min_prbs[sl] + additional[sl] for sl in min_prbs}
        assert sum(alloc.values()) <= self.total_prbs
        return alloc

    def _allocate_power(self, allocations):
        """
        Allocate transmit power using minimum guarantees + proportional sharing.
        Total power budget: Constants.MAX_POWER_W shared across slices.
        Returns: slice -> effective power in Watts
        """
        max_pw = self.max_power_w
        # Minimum power guarantees (Watts)
        min_pw = {"embb": 0.01 * max_pw, "urllc": 0.01 * max_pw}
        if self.enable_mmtc:
            min_pw["mmtc"] = 0.05 * max_pw

        gtotal = sum(min_pw.values())
        # Safety: if minimums exceed budget, scale down proportionally
        if gtotal > max_pw:
            scale  = max_pw / gtotal
            min_pw = {k: v * scale for k, v in min_pw.items()}
            gtotal = max_pw
        avail = max_pw - gtotal

        # Extract power requests from agent actions (index 1 = tx_power)
        requests = {"embb": allocations["embb_agent"][1], "urllc": allocations["urllc_agent"][1]}
        if self.enable_mmtc:
            requests["mmtc"] = allocations["mmtc_agent"][1]

        # Normalize requests to sum to 1
        total_req = sum(requests.values())
        requests  = ({k: v / total_req for k, v in requests.items()} if total_req > self.constants.EPS
                     else {k: 1.0 / len(requests) for k in requests})

        # Proportional allocation of remaining power budget
        extra = {k: v * avail for k, v in requests.items()}

        # Final allocation: guaranteed + proportional share (Watts)
        alloc = {sl: float(np.clip(min_pw[sl] + extra[sl], 0.0, max_pw)) for sl in min_pw}
        assert sum(alloc.values()) <= max_pw + 1e-3
        return alloc

    # ------------------------------------------------------------------
    # Traffic & channel dynamics (unchanged from original)
    # ------------------------------------------------------------------

    def _update_traffic_patterns(self):
        """
        Update time-varying traffic intensity using episode-randomized
        daily patterns with bounded noise.
        Traffic intensity is a dimensionless load multiplier.
        """
        # Convert step count to hour of day (288 steps = 24 hours)
        hour = (self.step_count % 288) / 12.0

        # eMBB traffic: high variability, strong diurnal pattern
        p = self.traffic_params["embb"]
        # For eMBB (allowing up to 5.0 multiplier = 250 Mbps absolute max safety cap)
        self.traffic_intensity["embb"] = float(np.clip(
            (p["base"] + p["amplitude"] * math.sin(2*math.pi*(hour-p["phase"])/24.0))
            * self._rng.uniform(0.9, 1.1), 0.1, 5.0))

        # URLLC traffic: lower variance, weaker diurnal pattern
        p = self.traffic_params["urllc"]
        # For URLLC (allowing up to 3.0 multiplier = 30 Mbps absolute max safety cap)
        self.traffic_intensity["urllc"] = float(np.clip(
            (p["base"] + p["amplitude"] * math.sin(2*math.pi*(hour-p["phase"])/24.0))
            * self._rng.uniform(0.9, 1.1), 0.05, 3.0)) # Changed from 1.0 to 3.0

        # mMTC traffic: periodic sensor reporting bursts
        if self.enable_mmtc:
            boost = 1.0 + self.constants.MMTC_PERIODIC_AMPLITUDE * (
                1 if self.step_count % self.constants.MMTC_PERIODIC_CYCLE == 0 else 0)
            self.traffic_intensity["mmtc"] = float(np.clip(
                self.constants.MMTC_BASE_RATE * boost * self._rng.uniform(0.9, 1.1), 5.0, 50.0))

    def _update_channel_conditions(self):
        """
        PHY-accurate channel model based on 3GPP TR 38.901 (UMa NLOS)
        """
        # From 3GPP UMa NLOS Standard
        SIGMA_DB = self.constants.SHADOWING_STD
        rho      = 0.9      # Temporal correlation factor
        DIST_MIN = 50.0     # meters (3GPP standard min is 10m, but 50m is safer for RL)
        DIST_MAX = 500.0    # meters

        fc = self.constants.CARRIER_FREQ_GHZ
        h_ut = self.constants.UE_HEIGHT_M

        for sl in self.channel_quality:
            if sl not in self.distance:
                self.distance[sl] = self._rng.uniform(DIST_MIN, DIST_MAX)
            elif self.enable_mobility:
                self.distance[sl] = float(np.clip(
                    self.distance[sl] + self._rng.normal(0, 1.0), DIST_MIN, DIST_MAX))

            d_3d = self.distance[sl]

            # ---------------------------------------------------------
            # 3GPP TR 38.901 UMa NLOS Pathloss Formula
            # PL = 13.54 + 39.08*log10(d) + 20*log10(fc) - 0.6*(h_ut - 1.5)
            # ---------------------------------------------------------
            pathloss_db = (
                13.54
                + 39.08 * np.log10(d_3d)
                + 20.0 * np.log10(fc)
                - 0.6 * (h_ut - 1.5)
            )

            # Correlated Log-normal shadowing (slice-level)
            z = self._rng.randn()
            self.shadowing[sl] = (rho * self.shadowing[sl]
                                  + np.sqrt(1 - rho**2) * SIGMA_DB * z)

            # Rayleigh small-scale fading
            h = (self._rng.randn() + 1j * self._rng.randn()) / np.sqrt(2)
            fading_db = 20.0 * np.log10(abs(h) + self.constants.EPS)

            # Final Channel Quality (Gain is negative pathloss)
            self.channel_quality[sl] = -pathloss_db + self.shadowing[sl] + fading_db

    # ------------------------------------------------------------------
    # PHY helpers
    # ------------------------------------------------------------------

    def _compute_shannon_capacity(self, prbs: int, sinr_db: float) -> float:
        """
        Compute Shannon capacity in Mbps.
        C = B * log2(1 + SINR)
        Where:
            B = total bandwidth (Hz)
            SINR in linear scale
        Returns:
            Capacity in Mbps (clipped for stability)
        """
        # No resources → no capacity
        if prbs <= 0:
            return 0.0

        # Clip SINR to realistic cellular range (prevents extreme exponentials)
        sinr_db     = np.clip(sinr_db, -10.0, 40.0)
        sinr_linear = 10.0 ** (sinr_db / 10.0)
        bandwidth   = prbs * self.bandwidth_per_prb

        # Safety cap to avoid reward explosion
        return float(np.clip(bandwidth * np.log2(1.0 + sinr_linear) / 1e6, 0.0, 1000.0))

    def _compute_bler(self, sinr_db: float, prbs: int) -> float:
        """
        Compute Block Error Rate (BLER) based on SINR and resource allocation
        Uses a sigmoid model:
        - Higher SINR -> Lower BLER
        - More PRBs provide diversity gain -> Lower BLER
        Returns: BLER in range [0, 1]
        """
        # SINR threshold for target BLER is usually ~5dB
        sinr_factor    = 1.0 / (1.0 + math.exp(0.5 * (sinr_db - 5.0)))
        # Diversity gain from multiple PRBs
        diversity_gain = 1.0 / (1.0 + 0.1 * prbs)

        return float(np.clip(sinr_factor * diversity_gain, 0.0, 1.0))

    # ------------------------------------------------------------------
    # NEW: centralised per-UE physics + queuing helper
    # ------------------------------------------------------------------

    def _compute_ue_metrics(
        self,
        slice_type:           str,
        ue_idx:               int,
        prbs_per_ue:          int,
        power_per_ue:         float,
        arrival_per_ue:       float,   # Mbps -- slice total / n_ues
        max_queue_per_ue:     float,   # Mbits -- slice buffer / n_ues
        tx_delay_ms:          float,
        processing_delay_ms:  float = 0.0,
        max_latency_cap:      float = 500.0,
        bler:                 float = 0.0,
    ) -> Dict[str, float]:
        """
        One-step PHY + fluid-queue update for a single UE.

        Traffic model
        -------------
        * The slice carries a FIXED total load (traffic_intensity x BASE_RATE).
        * Each UE receives:  arrival_per_ue = total_arrival / n_ues  (Mbps).
        * As n_ues increases, each UE's PRBs and power decrease -> service rate
          drops -> queue grows -> latency rises.  Total slice throughput (sum
          over UEs) stays bounded by slice capacity. This is the correct
          physical model: more UEs means resource contention, not more traffic.

        Per-UE independent fading
        -------------------------
        * channel_quality[slice_type] captures shared pathloss + shadowing.
        * An independent Rayleigh fade is drawn here for each UE, giving
          realistic SINR spread even among co-located UEs.

        Queuing model
        -------------
        * Discrete-time fluid queue (exact, no approximation):
            Q[t] = clip( Q[t-1] + lambda*dt - mu*dt , 0, Q_max )

        * Latency via Little's Law:
            W = Q / mu          (queuing delay only)
            L_total = W + tx_delay + processing_delay

        * Overflow packets are dropped (packet_loss > 0 only on overflow).
        """
        # ---- Independent per-UE Rayleigh fade ---------------------------
        # Draw complex gaussian to represent multi-path fading for this specific UE
        h            = (self._rng.randn() + 1j * self._rng.randn()) / np.sqrt(2)
        ue_fading_db = 20.0 * np.log10(abs(h) + self.constants.EPS)

        # Combine with slice-level channel (pathloss + shadowing)
        ue_channel_db = self.channel_quality[slice_type] + ue_fading_db

        # ---- SINR -------------------------------------------------------
        # Calculate final SINR using Thermal Noise formula
        NOISE_FIGURE_DB = 5.0
        bandwidth_hz    = prbs_per_ue * self.bandwidth_per_prb
        noise_dbm       = -174.0 + 10.0 * math.log10(bandwidth_hz) + NOISE_FIGURE_DB
        tx_power_dbm    = 10.0 * math.log10(power_per_ue * 1000.0)
        sinr_db         = ue_channel_db + tx_power_dbm - noise_dbm

        # ---- Shannon capacity -------------------------------------------
        capacity_mbps      = self._compute_shannon_capacity(prbs_per_ue, sinr_db)
        effective_capacity = capacity_mbps * (1.0 - bler)

        # ---- Fluid queue update -----------------------------------------
        #
        #  bits arrived  = arrival_per_ue [Mbps] * 1e6 * dt [s]
        #  bits served   = effective_capacity [Mbps] * 1e6 * dt [s]
        #  new_queue     = prev_queue + arrived - served  (clipped to [0, Q_max])
        #
        bits_arrived = arrival_per_ue      * 1e6 * self.constants.TIMESTEP_SEC
        bits_served  = effective_capacity  * 1e6 * self.constants.TIMESTEP_SEC
        max_q_bits   = max_queue_per_ue * 1e6

        prev_q_bits = self.ue_queue_lengths[slice_type][ue_idx] * 1e6
        new_q_bits  = prev_q_bits + bits_arrived - bits_served

        if new_q_bits > max_q_bits:
            # Buffer overflow: drop excess bits
            overflow_bits = new_q_bits - max_q_bits
            packet_loss   = min(1.0, bler + overflow_bits / (bits_arrived + self.constants.EPS))
            self.ue_queue_lengths[slice_type][ue_idx] = max_q_bits / 1e6
        elif new_q_bits < 0:
            packet_loss = bler
            self.ue_queue_lengths[slice_type][ue_idx] = 0.0
        else:
            packet_loss = bler
            self.ue_queue_lengths[slice_type][ue_idx] = new_q_bits / 1e6

        # ---- Latency: Little's Law + fixed delays -----------------------
        #
        #  W (queuing delay) = Q / mu   [Little's Law for fluid queue]
        #
        q_mb = self.ue_queue_lengths[slice_type][ue_idx]
        if effective_capacity > self.constants.EPS:
            queuing_delay_ms = (q_mb / effective_capacity) * 1000.0
        else:
            queuing_delay_ms = max_latency_cap if q_mb > self.constants.EPS else 0.0

        latency = min(processing_delay_ms + tx_delay_ms + queuing_delay_ms,
                      max_latency_cap)

        # ---- Goodput (bounded by both capacity and demand) --------------
        throughput = min(effective_capacity, arrival_per_ue)

        # ---- Spectral efficiency ----------------------------------------
        se = (capacity_mbps * 1e6) / bandwidth_hz if bandwidth_hz > 0 else 0.0

        return {
            "throughput":          throughput,
            "latency":             latency,
            "packet_loss":         packet_loss,
            "queue_length":        self.ue_queue_lengths[slice_type][ue_idx],
            "spectral_efficiency": se,
        }

    # ------------------------------------------------------------------
    # NEW: aggregation helper
    # ------------------------------------------------------------------

    def _aggregate_slice_metrics(
        self, slice_type: str, ue_results: List[Dict[str, float]]
    ):
        """
        Collapse per-UE metric dicts into slice-level metrics and update history.

        Aggregation rules
        -----------------
        * throughput, queue_length  ->  sum   (additive physical quantities)
        * latency, packet_loss,
          spectral_efficiency       ->  mean  (representative UE experience)
        """
        self.metrics[slice_type]["throughput"]          = float(sum(r["throughput"]          for r in ue_results))
        self.metrics[slice_type]["queue_length"]        = float(sum(r["queue_length"]        for r in ue_results))
        self.metrics[slice_type]["latency"]             = float(np.mean([r["latency"]             for r in ue_results]))
        self.metrics[slice_type]["packet_loss"]         = float(np.mean([r["packet_loss"]         for r in ue_results]))
        self.metrics[slice_type]["spectral_efficiency"] = float(np.mean([r["spectral_efficiency"] for r in ue_results]))

        for key in self.metrics[slice_type]:
            self.metric_history[slice_type][key].append(self.metrics[slice_type][key])

    # ------------------------------------------------------------------
    # Slice update methods (thin wrappers around _compute_ue_metrics)
    # ------------------------------------------------------------------

    def _update_embb_slice(self, prbs: int, power_level: float):
        """
        Update eMBB slice metrics with multi-UE support.
        eMBB: high-throughput, moderate-latency slice.

        * Total slice traffic = traffic_intensity x EMBB_BASE_RATE_MBPS (Mbps).
        * This is constant regardless of UE count. Each UE receives an equal
          share of traffic, PRBs, and power.
        * Per-UE computations are dispatched to `_compute_ue_metrics`
        * Final metrics are aggregated back to the slice level
        """
        n_ues = self.num_ues["embb"]

        # Fixed total slice traffic -- does NOT scale with n_ues
        total_arrival_mbps = (self.traffic_intensity["embb"]
                              * self.constants.EMBB_BASE_RATE_MBPS)

        prbs_per_ue    = max(1, prbs // n_ues)
        power_per_ue   = power_level / n_ues
        arrival_per_ue = total_arrival_mbps #/ n_ues   # equal traffic share
        max_q_per_ue   = self.embb_max_queue  / n_ues  # equal buffer share

        results = [
            self._compute_ue_metrics(
                slice_type="embb",
                ue_idx=i,
                prbs_per_ue=prbs_per_ue,
                power_per_ue=power_per_ue,
                arrival_per_ue=arrival_per_ue,
                max_queue_per_ue=max_q_per_ue,
                tx_delay_ms=self.constants.EMBB_TRANSMISSION_DELAY_MS,
                max_latency_cap=500.0,
            )
            for i in range(n_ues)
        ]
        self._aggregate_slice_metrics("embb", results)

    def _update_urllc_slice(self, prbs: int, power_level: float):
        """
        Update URLLC slice metrics with multi-UE support.
        URLLC: ultra-low-latency, high-reliability slice.

        * Total slice traffic = traffic_intensity x URLLC_BASE_RATE_MBPS (Mbps).
        * With more UEs, service rate per UE drops -> queueing latency rises
          -> harder to meet the 1 ms SLA. This is the intended behaviour.
        * Per-UE computations are dispatched to `_compute_ue_metrics`
        * Final metrics are aggregated back to the slice level
        """
        n_ues = self.num_ues["urllc"]

        total_arrival_mbps = (self.traffic_intensity["urllc"]
                              * self.constants.URLLC_BASE_RATE_MBPS)

        prbs_per_ue    = max(1, prbs // n_ues)
        power_per_ue   = power_level / n_ues
        arrival_per_ue = total_arrival_mbps #/ n_ues
        max_q_per_ue   = self.urllc_max_queue / n_ues

        results = [
            self._compute_ue_metrics(
                slice_type="urllc",
                ue_idx=i,
                prbs_per_ue=prbs_per_ue,
                power_per_ue=power_per_ue,
                arrival_per_ue=arrival_per_ue,
                max_queue_per_ue=max_q_per_ue,
                tx_delay_ms=self.constants.URLLC_TRANSMISSION_DELAY_MS,
                processing_delay_ms=self.constants.URLLC_PROCESSING_DELAY_MS,
                max_latency_cap=1000.0,
                bler=0.0,
            )
            for i in range(n_ues)
        ]
        self._aggregate_slice_metrics("urllc", results)

    def _update_mmtc_slice(self, prbs: int, power_level: float):
        """
        Update mMTC slice metrics (massive IoT).
        mMTC uses a device-density model; UE count sets how many IoT
        sub-groups share the slice resources equally.
        """
        if not self.enable_mmtc:
            return

        n_ues        = self.num_ues["mmtc"]
        prbs_per_ue  = max(1, prbs // n_ues)
        power_per_ue = power_level / n_ues
        num_devices  = int(self.traffic_intensity["mmtc"] * 1000)

        # Aggregate across UE sub-groups
        total_connected  = 0
        total_throughput = 0.0
        total_epb        = 0.0

        for _ in range(n_ues):
            # Independent per-group Rayleigh fade
            h            = (self._rng.randn() + 1j * self._rng.randn()) / np.sqrt(2)
            ue_fading_db = 20.0 * np.log10(abs(h) + self.constants.EPS)
            eff_sinr     = self.channel_quality["mmtc"] + ue_fading_db + 10 * math.log10(power_per_ue + 0.01)
            capacity     = self._compute_shannon_capacity(prbs_per_ue, eff_sinr)

            devs_here = num_devices // n_ues
            # 10 devices/PRB simplified model
            connected = min(devs_here, prbs_per_ue * 10)

            # Throughput capped at 0.01 Mbps per device
            grp_tput  = min(connected * 0.01, capacity)
            bits_tx   = grp_tput * 1e6
            epb       = (power_per_ue * 10) / bits_tx if bits_tx > 0 else 1000.0

            total_connected  += connected
            total_throughput += grp_tput
            total_epb        += epb

        self.metrics["mmtc"]["connection_density"] = (
            total_connected / num_devices if num_devices > 0 else 1.0)
        self.metrics["mmtc"]["active_devices"]    = float(total_connected)
        self.metrics["mmtc"]["throughput"]         = float(total_throughput)
        self.metrics["mmtc"]["latency"]            = 100 + self._rng.exponential(50)
        self.metrics["mmtc"]["energy_efficiency"]  = float(total_epb / n_ues)

        for key in self.metrics["mmtc"]:
            self.metric_history["mmtc"][key].append(self.metrics["mmtc"][key])

    # ------------------------------------------------------------------
    # Reward & cost computation (restored old explanations)
    # ------------------------------------------------------------------

    def _compute_rewards(self, allocations):
        """
        Compute per-agent rewards with normalized objectives and
        a small cooperative (global) component.
        """
        # ============================================================
        # eMBB reward (throughput-driven, latency-aware)
        # ============================================================
        embb_tp_norm = np.clip(
            self.metrics["embb"]["throughput"] / self.constraints["embb"]["min_throughput"],
            0.0, 2.0) / 2.0

        embb_lat_norm = np.clip(
            self.metrics["embb"]["latency"] / self.constraints["embb"]["max_latency"],
            0.0, 2.0) / 2.0

        # Mix throughput and latency, then map to [-1, 1]
        embb_reward = 2.0 * (0.65 * embb_tp_norm + 0.35 * (1.0 - embb_lat_norm)) - 1.0

        # ============================================================
        # URLLC reward (SLA-anchored, smooth, tail-critical)
        # ============================================================

        # -------------------------
        # Targets (explicit SLA)
        # -------------------------
        MEAN_LAT_TARGET = 250.0      # ms

        latency     = self.metrics["urllc"]["latency"]
        packet_loss = self.metrics["urllc"]["packet_loss"]

        # Tail latency proxy (p95 from history)
        recent_lat   = list(self.metric_history["urllc"]["latency"])
        tail_latency = np.percentile(recent_lat, 95) if len(recent_lat) >= 5 else latency

        # 1. Mean latency reward (sharp around 1 ms)
        # ≈1 at 1ms, drops fast if >1.2ms
        mean_lat_rwd = np.exp(-1 * abs(latency / MEAN_LAT_TARGET - 1.0))

        # 2. Tail latency reward (hard requirement < 1 ms)
        tail_lat_rwd = np.exp(-2 * max(0.0, tail_latency / MEAN_LAT_TARGET - 1.0))

        # 3. Reliability reward (discoverable but strict)
        reliability = 1.0 - packet_loss
        KNEE, TARGET, BELOW_MAX = 0.999, 0.99999, 0.6
        if reliability < KNEE:
            # Linear encouragement — capped at 0.6, never reaches 1.0
            rel_rwd = (reliability / KNEE) * BELOW_MAX
        else:
            # Picks up exactly at 0.6, curves up to 1.0 at 99.999%
            rel_rwd = BELOW_MAX + (1.0 - BELOW_MAX) * (
                1 - np.exp(-5.0 * (reliability - KNEE) / (TARGET - KNEE)))
        rel_rwd = min(rel_rwd, 1.0)

        # 4. Core URLLC reward (multiplicative = AND logic)
        # All three must be satisfied simultaneously, mapped to [-1, 1]
        core_reward   = 2.0 * (mean_lat_rwd * tail_lat_rwd * rel_rwd) - 1.0

        # 5. Throughput & resource efficiency (secondary)
        urllc_tp_norm = np.clip(
            self.metrics["urllc"]["throughput"] / self.constraints["urllc"]["min_throughput"],
            0.0, 1.0)
        res_eff = 1.0 - self.actual_prb_allocation["urllc"] / self.total_prbs

        # 6. Final URLLC reward
        # Core (latency + reliability) dominant at 0.8
        urllc_reward = (0.8 * core_reward
                        + 0.1 * (2.0 * urllc_tp_norm - 1.0)
                        + 0.1 * (2.0 * res_eff - 1.0))

        # ============================================================
        # Global cooperative reward (URLLC latency + eMBB throughput)
        # ============================================================
        embb_tp_norm2 = np.clip(
            self.metrics["embb"]["throughput"] / self.constraints["embb"]["min_throughput"],
            0.0, 1.0)

        URLLC_MIN = (self.constants.URLLC_PROCESSING_DELAY_MS
                     + self.constants.URLLC_TRANSMISSION_DELAY_MS) # = 1.0ms

        lat_tgt = self.constraints["urllc"]["max_latency"]

        # Closer to 1ms = better, at 1ms = 1.0, beyond = negative
        if self.metrics["urllc"]["latency"] <= lat_tgt:
            u_lat_rwd = 1.0
        else:
            u_lat_rwd = -np.clip((self.metrics["urllc"]["latency"] - URLLC_MIN) / URLLC_MIN,
                                 0.0, 1.0)

        # Combine — URLLC latency dominant. Map to [-1, 1]
        global_reward = 2.0 * (0.7 * u_lat_rwd + 0.3 * embb_tp_norm2) - 1.0

        # ============================================================
        # Mix individual + global rewards
        # ============================================================
        alpha = 0.30
        rewards = {
            "embb_agent":  (1 - alpha) * embb_reward  + alpha * global_reward,
            "urllc_agent": (1 - alpha) * urllc_reward + alpha * global_reward,
        }
        if self.enable_mmtc:
            mmtc_reward = self.metrics["mmtc"]["connection_density"] * 2.0 - 1.0
            rewards["mmtc_agent"] = (1 - alpha) * mmtc_reward + alpha * global_reward

        return rewards

    def _compute_costs(self):
        """
        Compute normalized constraint violation costs in [0, 1].
        Cost = 0  → all constraints satisfied
        Cost > 0  → degree of SLA violation

        Design principle: costs are slice-local only.
        Cross-slice pressure emerges naturally from shared PRB/power
        budget — no artificial coupling needed here.
        """
        def viol(val, thr, is_min=False):
            """
            Log-scaled violation in [0, 1].
            is_min=False → value should be ≤ threshold (e.g. latency)
            is_min=True  → value should be ≥ threshold (e.g. throughput)
            Returns 0.0 when the constraint is satisfied.
            """
            eps = self.constants.EPS
            if is_min:
                if val >= thr: return 0.0
                ratio = thr / (val + eps)
            else:
                if val <= thr: return 0.0
                ratio = val / (thr + eps) # ← guard zero-threshold
            return float(np.clip(np.log10(ratio), 0.0, 1.0))

        # Simple mean keeps each term on the same [0,1] scale
        costs = {
            "embb":  float(np.clip((
                viol(self.metrics["embb"]["latency"],    self.constraints["embb"]["max_latency"]) +
                viol(self.metrics["embb"]["throughput"], self.constraints["embb"]["min_throughput"], True)
            ) / 2, 0.0, 1.0)),
            "urllc": float(np.clip((
                viol(self.metrics["urllc"]["latency"],     self.constraints["urllc"]["max_latency"]) +
                viol(self.metrics["urllc"]["packet_loss"], self.constraints["urllc"]["max_packet_loss"])
            ) / 2, 0.0, 1.0)),
        }
        if self.enable_mmtc:
            costs["mmtc"] = float(np.clip((
                viol(self.metrics["mmtc"]["connection_density"], self.constraints["mmtc"]["min_connection_density"], True) +
                viol(self.metrics["mmtc"]["energy_efficiency"],  self.constraints["mmtc"]["max_energy_per_bit"])
            ) / 2, 0.0, 1.0))
        return costs

    # ------------------------------------------------------------------
    # Observation builder (unchanged from original)
    # ------------------------------------------------------------------

    def _get_observation(self, agent):
        """
        Build observation vector for a given agent.

        Layout (15 dims):
        [ current(5), historical_avg(5), channel_current, channel_trend,
          traffic_intensity, time_sin, time_cos ]
        """
        sl = self.agent_to_slice[agent]

        # 1. Current metrics (5)
        if sl == "mmtc":
            current = np.array([
                self.metrics[sl]["throughput"],
                self.metrics[sl]["latency"],
                self.metrics[sl]["connection_density"] * 100.0,
                self.metrics[sl]["energy_efficiency"],
                self.metrics[sl]["active_devices"] / 1000.0,
            ], dtype=np.float32)
        else:
            current = np.array([
                self.metrics[sl]["throughput"],
                self.metrics[sl]["latency"],
                self.metrics[sl]["packet_loss"] * 1000.0,
                self.metrics[sl]["queue_length"],
                self.metrics[sl]["spectral_efficiency"],
            ], dtype=np.float32)

        # 2. Historical averages (5) (warm-started, always safe)
        historical = np.array([
            np.mean(list(self.metric_history[sl][k])[-10:])
            for k in list(self.metrics[sl].keys())[:5]
        ], dtype=np.float32)

        # 3. Channel quality (current + trend)
        ch_trend = 0.0
        if len(self.metric_history[sl]["throughput"]) > 5:
            ch_trend = float(np.mean(np.diff(
                list(self.metric_history[sl]["throughput"])[-5:])))

        channel  = np.array([self.channel_quality[sl], ch_trend], dtype=np.float32)

        # 4. Traffic intensity (1)
        traffic  = np.array([self.traffic_intensity[sl]], dtype=np.float32)

        # 5. Time encoding (2) – cyclic
        hour     = (self.step_count % 288) / 12.0
        time_enc = np.array([math.sin(2.0*math.pi*hour/24.0),
                              math.cos(2.0*math.pi*hour/24.0)], dtype=np.float32)

        # CRITICAL: Clip to declared observation space
        obs = np.concatenate([current, historical, channel, traffic, time_enc])
        return np.clip(obs, self._observation_spaces[agent].low,
                       self._observation_spaces[agent].high)

    # ------------------------------------------------------------------
    # Info, state, render
    # ------------------------------------------------------------------

    def _build_infos(self, costs):
        """Build info dictionaries for each agent"""
        infos = {
            "embb_agent":  {"cost": costs["embb"],  "metrics": self.metrics["embb"].copy(),
                            "constraints": self.constraints["embb"],
                            "num_ues": self.num_ues["embb"]},
            "urllc_agent": {"cost": costs["urllc"], "metrics": self.metrics["urllc"].copy(),
                            "constraints": self.constraints["urllc"],
                            "num_ues": self.num_ues["urllc"]},
        }
        if self.enable_mmtc:
            infos["mmtc_agent"] = {
                "cost": costs["mmtc"], "metrics": self.metrics["mmtc"].copy(),
                "constraints": self.constraints["mmtc"],
                "num_ues": self.num_ues["mmtc"],
            }
        return infos

    def state(self):
        """Return global state (for centralized training)"""
        sl_list = []
        for sl in ["embb", "urllc"] + (["mmtc"] if self.enable_mmtc else []):
            sl_list.extend(self.metrics[sl].values())
        sl_list.extend(self.channel_quality.values())
        sl_list.extend(self.traffic_intensity.values())
        sl_list.append(self.step_count / self.max_steps)
        return np.array(sl_list, dtype=np.float32)

    def render(self):
        """Render current state (for debugging)"""
        print(f"\n{'='*60}")
        print(f"Step: {self.step_count}/{self.max_steps}")
        print(f"{'='*60}")
        for sl in ["embb", "urllc"] + (["mmtc"] if self.enable_mmtc else []):
            n = self.num_ues[sl]
            print(f"\n{sl.upper()} Slice  [{n} UE{'s' if n > 1 else ''}]:")
            print(f"  Channel Quality (slice):  {self.channel_quality[sl]:.2f} dB")
            print(f"  Traffic Intensity:        {self.traffic_intensity[sl]:.3f}")
            print(f"  Slice-level aggregates:")
            for k, v in self.metrics[sl].items():
                print(f"    {k:25s}: {v:.4f}")
            if sl in self.ue_queue_lengths:
                print(f"  Per-UE queue [Mbits]:     {[round(q,4) for q in self.ue_queue_lengths[sl]]}")
        print(f"\n{'='*60}\n")

    def close(self):
        """Cleanup"""
        pass

## Constrained MAPPO Trainer for RAN Slicing

This module implements **Multi-Agent PPO (MAPPO)** with **Lagrangian constraint
handling** to train the `embb_agent` and `urllc_agent` policies from the RAN
slicing environment defined above.

### Architecture

- **`ActorCritic`** — a shared network class with:
  - An **actor** that maps local observations → a Beta-like continuous action
    distribution (`Normal`, mean squashed via sigmoid, learnable log-std), used
    to sample `[PRB_fraction, tx_power_level]`.
  - A **critic** that maps the *global state* (not local obs) to a value
    estimate — this is the "centralized training, decentralized execution"
    pattern standard to MAPPO.
  - A **cost critic**, structurally identical to the value critic, that
    estimates expected constraint-violation cost rather than reward.

- **`MAPPO`** — wraps `ActorCritic` with:
  - **GAE** (Generalized Advantage Estimation) computed twice per rollout: once
    for the Lagrangian-augmented reward (`reward - λ * cost`) and once for the
    raw cost signal.
  - **Clipped PPO objective** extended with a cost-advantage penalty term, so
    the policy is discouraged from increasing expected constraint violations.
  - Separate clipped value losses for both the reward critic and the cost
    critic.

### Constrained RL / Lagrangian Method

A **dual variable `λ`** is updated after every episode based on how far the
average constraint cost is from a target `cost_limit`:

λ ← clip(λ + lambda_lr * (avg_cost - cost_limit), 0, lambda_max)


This lets the reward signal automatically trade off performance vs. constraint
satisfaction — no fixed penalty weight needs to be hand-tuned.

### `TrainingLogger`

A lightweight metrics tracker (unbounded `deque`s) that:
- Records per-episode rewards, costs, λ, violation rates/magnitudes, and
  detailed per-slice PRB/power/latency/throughput/queue statistics.
- Prints a formatted console summary every `eval_freq` episodes
  (`print_summary`).
- Produces a 4×3 grid of training curves — rewards, cost/λ, violation rate,
  per-slice throughput/latency/reliability/PRB-efficiency, and resource
  allocation over time — via `plot_training`, saved to disk as a PNG.

### `train_constrained_marl(...)`

The main training loop:
1. Builds the environment (5 UEs per slice, mobility and time-varying traffic
   enabled, mMTC disabled by default).
2. Instantiates one `MAPPO` agent per slice (`embb_agent`, `urllc_agent`).
3. For each episode: rolls out a full trajectory, computes a blended system
   cost (`0.3 * eMBB_cost + 0.7 * URLLC_cost`), stores transitions per agent,
   updates both policies via PPO, and updates the Lagrangian dual variable.
4. Logs detailed metrics every episode and prints a summary every `eval_freq`
   episodes.
5. Saves periodic and final checkpoints (agent weights + dual variable) to
   `save_dir`, and produces final training plots.

**Configuration note:** `save_dir` defaults to a local `./checkpoints` folder
so the script runs out-of-the-box on any machine. If you're running this on
Google Colab with Drive mounted, pass your own path, e.g.
`save_dir="/content/drive/MyDrive/your_folder"`.

**Entry point:** running this file directly (`python train_marl.py`) launches
a full 15,000-episode training run with the hyperparameters shown in the
`if __name__ == "__main__":` block — adjust these as needed for your own
experiments.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import numpy as np
from typing import Tuple, List
import matplotlib.pyplot as plt
from collections import deque
import os

class ActorCritic(nn.Module):
    """Actor-Critic network for MAPPO"""

    def __init__(self, obs_dim: int, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super().__init__()

        self.obs_dim = obs_dim
        self.state_dim = state_dim

        # Actor (policy) - uses local observation
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.actor_mean = nn.Linear(hidden_dim, action_dim)
        self.actor_logstd = nn.Parameter(torch.zeros(action_dim))

        # Critic (value) - uses global state for centralized training
        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

        # Cost critic (for constraint)
        self.cost_critic = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def get_action_distribution(self, obs: torch.Tensor):
        """Get action distribution from observation"""
        action_hidden = self.actor(obs)
        action_mean = torch.sigmoid(self.actor_mean(action_hidden))
        action_std = torch.exp(self.actor_logstd).clamp(0.01, 1.0)
        return action_mean, action_std

    def get_action(self, obs: np.ndarray):
        """Sample action from policy - ONLY USES OBSERVATION"""
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs).unsqueeze(0)

            action_mean, action_std = self.get_action_distribution(obs_t)

            dist = Normal(action_mean, action_std)
            action = dist.sample()
            action = torch.clamp(action, 0.0, 1.0)
            log_prob = dist.log_prob(action).sum(-1)

        return action.squeeze(0).numpy(), log_prob.item()

    def evaluate(self, obs: torch.Tensor, state: torch.Tensor, action: torch.Tensor):
        """Evaluate actions - uses obs for actor, state for critics"""
        # Get action distribution from observation
        action_mean, action_std = self.get_action_distribution(obs)

        dist = Normal(action_mean, action_std)
        log_prob = dist.log_prob(action).sum(-1, keepdim=True)
        entropy = dist.entropy().sum(-1, keepdim=True)

        # Get values from state
        value = self.critic(state)
        cost_value = self.cost_critic(state)

        return log_prob, entropy, value, cost_value


class MAPPO:
    """Multi-Agent PPO with Lagrangian constraint handling"""

    def __init__(
        self,
        obs_dim: int,
        state_dim: int,
        action_dim: int = 2,
        hidden_dim: int = 128,
        lr: float = 3e-4,
        gamma: float = 0.99,
        gae_lambda: float = 0.95,
        clip_epsilon: float = 0.2,
        c1: float = 0.5,  # Value loss coefficient
        c2: float = 0.01,  # Entropy coefficient
        epochs: int = 10,
        batch_size: int = 64,
        device: str = "cpu"
    ):
        self.obs_dim = obs_dim
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.c1 = c1
        self.c2 = c2
        self.epochs = epochs
        self.batch_size = batch_size
        self.device = device

        # Network
        self.ac = ActorCritic(obs_dim, state_dim, action_dim, hidden_dim).to(device)
        self.optimizer = optim.Adam(self.ac.parameters(), lr=lr)

        # For tracking
        self.training_stats = {
            "policy_loss": [],
            "value_loss": [],
            "cost_value_loss": [],
            "entropy": []
        }

    def select_action(self, obs: np.ndarray):
        """Select action using current policy"""
        return self.ac.get_action(obs)

    def compute_gae(
        self,
        rewards: List[float],
        costs: List[float],
        dones: List[bool],
        values: torch.Tensor,
        cost_values: torch.Tensor,
        lambda_dual: float
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """Compute GAE for both rewards and costs"""

        # Convert to tensors
        rewards = torch.FloatTensor(rewards).to(self.device)
        costs = torch.FloatTensor(costs).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)

        # Augmented reward (Lagrangian)
        augmented_rewards = rewards - lambda_dual * costs

        # GAE for augmented reward
        advantages = torch.zeros_like(rewards)
        gae = 0

        for t in reversed(range(len(rewards))):
            if t == len(rewards) - 1:
                next_value = 0
            else:
                next_value = values[t + 1]

            delta = augmented_rewards[t] + self.gamma * next_value * (1 - dones[t]) - values[t]
            gae = delta + self.gamma * self.gae_lambda * (1 - dones[t]) * gae
            advantages[t] = gae

        # Returns
        returns = advantages + values

        # GAE for costs (for cost critic AND policy loss)
        cost_advantages = torch.zeros_like(costs)
        gae_cost = 0

        for t in reversed(range(len(costs))):
            if t == len(costs) - 1:
                next_cost_value = 0
            else:
                next_cost_value = cost_values[t + 1]

            delta_cost = costs[t] + self.gamma * next_cost_value * (1 - dones[t]) - cost_values[t]
            gae_cost = delta_cost + self.gamma * self.gae_lambda * (1 - dones[t]) * gae_cost
            cost_advantages[t] = gae_cost

        cost_returns = cost_advantages + cost_values

        return advantages, returns, cost_advantages, cost_returns

    def update(self, data: Tuple, lambda_dual: float = 0.0):
        """Update policy using PPO with cost constraints"""

        obs_list, states_list, acts_list, logps_list, rews_list, costs_list, dones_list = data

        # Convert to tensors
        obs = torch.FloatTensor(np.array(obs_list)).to(self.device)
        states = torch.FloatTensor(np.array(states_list)).to(self.device)
        actions = torch.FloatTensor(np.array(acts_list)).to(self.device)
        old_log_probs = torch.FloatTensor(logps_list).unsqueeze(1).to(self.device)

        # Get values for GAE computation
        with torch.no_grad():
            _, _, values, cost_values = self.ac.evaluate(obs, states, actions)
            # Store old values for clipping
            old_values = values.clone()
            old_cost_values = cost_values.clone()

        # Compute advantages and returns
        advantages, returns, cost_advantages, cost_returns = self.compute_gae(
            rews_list, costs_list, dones_list,
            values.squeeze(), cost_values.squeeze(),
            lambda_dual
        )

        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        if cost_advantages.std() > 1e-6:
            cost_advantages = (cost_advantages - cost_advantages.mean()) / (cost_advantages.std() + 1e-8)

        # PPO update
        for epoch in range(self.epochs):
            # Create mini-batches
            indices = np.arange(len(obs_list))
            np.random.shuffle(indices)

            for start in range(0, len(indices), self.batch_size):
                end = min(start + self.batch_size, len(indices))
                batch_idx = indices[start:end]

                # Get batch data
                obs_batch = obs[batch_idx]
                states_batch = states[batch_idx]
                actions_batch = actions[batch_idx]
                old_log_probs_batch = old_log_probs[batch_idx]
                advantages_batch = advantages[batch_idx].unsqueeze(1)
                returns_batch = returns[batch_idx].unsqueeze(1)
                cost_returns_batch = cost_returns[batch_idx].unsqueeze(1)
                old_values_batch = old_values[batch_idx]
                old_cost_values_batch = old_cost_values[batch_idx]
                cost_advantages_batch = cost_advantages[batch_idx].unsqueeze(1)

                # Evaluate actions
                log_probs, entropy, values, cost_values = self.ac.evaluate(
                    obs_batch, states_batch, actions_batch
                )

                # Policy loss with cost advantage
                ratio = torch.exp(log_probs - old_log_probs_batch)
                surr1 = ratio * advantages_batch
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages_batch

                # Add cost term to policy objective
                policy_loss = -(
                    torch.min(surr1, surr2) - lambda_dual * cost_advantages_batch
                ).mean()

                # Value loss with clipping
                value_pred_clipped = old_values_batch + torch.clamp(
                    values - old_values_batch,
                    -self.clip_epsilon,
                    self.clip_epsilon
                )
                value_loss = 0.5 * torch.max(
                    (returns_batch - values).pow(2),
                    (returns_batch - value_pred_clipped).pow(2)
                ).mean()

                # Cost value loss with clipping
                cost_value_pred_clipped = old_cost_values_batch + torch.clamp(
                    cost_values - old_cost_values_batch,
                    -self.clip_epsilon,
                    self.clip_epsilon
                )
                cost_value_loss = 0.5 * torch.max(
                    (cost_returns_batch - cost_values).pow(2),
                    (cost_returns_batch - cost_value_pred_clipped).pow(2)
                ).mean()

                # Entropy bonus
                entropy_loss = -entropy.mean()

                # Total loss
                loss = policy_loss + self.c1 * (value_loss + cost_value_loss) + self.c2 * entropy_loss

                # Optimization step
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.ac.parameters(), 0.5)
                self.optimizer.step()

                # Track stats
                self.training_stats["policy_loss"].append(policy_loss.item())
                self.training_stats["value_loss"].append(value_loss.item())
                self.training_stats["cost_value_loss"].append(cost_value_loss.item())
                self.training_stats["entropy"].append(-entropy_loss.item())

class TrainingLogger:
    """Logger for tracking training metrics - COMPLETE VERSION"""

    def __init__(self, window_size: int = 50):
        self.window_size = window_size
        self.metrics = {
            # Rewards & Constraints
            "episode_rewards": {"embb": deque(maxlen=None), "urllc": deque(maxlen=None)},
            "episode_costs": deque(maxlen=None),
            "lambda_dual": deque(maxlen=None),

            # Violations
            "constraint_violations": deque(maxlen=None),
            "violation_magnitude": deque(maxlen=None),
            "urllc_violation_magnitude": deque(maxlen=None),
            "embb_violation_magnitude": deque(maxlen=None),

            # eMBB Performance
            "embb_throughput": deque(maxlen=None),
            "embb_spectral_eff": deque(maxlen=None),
            "embb_packet_loss": deque(maxlen=None),
            "embb_power": deque(maxlen=None),
            "embb_prb_eff": deque(maxlen=None),
            "embb_queue": deque(maxlen=None),

            # URLLC Performance
            "urllc_latency": deque(maxlen=None),
            "urllc_latency_p95": deque(maxlen=None),
            "urllc_latency_max": deque(maxlen=None),
            "urllc_packet_loss": deque(maxlen=None),
            "urllc_reliability": deque(maxlen=None),
            "urllc_power": deque(maxlen=None),
            "urllc_prb_eff": deque(maxlen=None),
            "urllc_queue": deque(maxlen=None),

            # PRB Allocation
            "prb_requested": {"embb": deque(maxlen=None), "urllc": deque(maxlen=None)},
            "prb_assigned": {"embb": deque(maxlen=None), "urllc": deque(maxlen=None)},
        }

    def log_episode(self, ep: int, metrics: dict):
        """Log episode metrics"""
        for key, value in metrics.items():
            if key in ["episode_rewards", "prb_requested", "prb_assigned"]:
                # Nested dict (per agent)
                for agent, val in value.items():
                    self.metrics[key][agent].append(val)
            else:
                # Single value
                self.metrics[key].append(value)

    def print_summary(self, ep: int):
        """Print comprehensive training summary"""
        print(f"\n{'='*80}")
        print(f"Episode {ep:4d} Summary")
        print(f"{'='*80}")

        if len(self.metrics['episode_rewards']['embb']) == 0:
            print("No data yet.")
            return

        # Get recent data
        def get_recent(key):
            return list(self.metrics[key])[-self.window_size:]

        def get_recent_agent(key, agent):
            return list(self.metrics[key][agent])[-self.window_size:]

        # === REWARDS & CONSTRAINTS ===
        print(f"\n{'Rewards & Constraints':-^80}")
        recent_embb_r = get_recent_agent('episode_rewards', 'embb')
        recent_urllc_r = get_recent_agent('episode_rewards', 'urllc')
        recent_costs = get_recent('episode_costs')
        recent_lambda = get_recent('lambda_dual')
        recent_violations = get_recent('constraint_violations')

        print(f"  Episode Rewards (last {self.window_size}):")
        print(f"    eMBB:  {np.mean(recent_embb_r):8.2f} \u00b1 {np.std(recent_embb_r):6.2f}")
        print(f"    URLLC: {np.mean(recent_urllc_r):8.2f} \u00b1 {np.std(recent_urllc_r):6.2f}")
        print(f"  Constraint Cost:  {np.mean(recent_costs):8.4f} \u00b1 {np.std(recent_costs):6.4f}")
        print(f"  Lambda (dual):    {np.mean(recent_lambda):8.4f}")

        # ==== Violations ====
        print(f"\n{'Violations':-^80}")
        print(f"  Violation Frequency:           {np.mean(recent_violations):8.2f}%")
        print(f"  Violation Magnitude (System):  {100 * np.mean(get_recent('violation_magnitude')):8.4f}%")
        print(f"  Violation Magnitude (URLLC):   {100 * np.mean(get_recent('urllc_violation_magnitude')):8.4f}%")
        print(f"  Violation Magnitude (eMBB):    {100 * np.mean(get_recent('embb_violation_magnitude')):8.4f}%")

        # === eMBB PERFORMANCE ===
        print(f"\n{'eMBB Performance (5 UEs)':-^80}")
        print(f"  Throughput:         {np.mean(get_recent('embb_throughput')):8.2f} Mbps  "
              f"(target: \u226550 Mbps)")
        print(f"  Spectral Eff:       {np.mean(get_recent('embb_spectral_eff')):8.2f} bps/Hz")
        print(f"  Queue Length:       {np.mean(get_recent('embb_queue')):8.2f} Mbits")
        print(f"  Packet Loss:        {np.mean(get_recent('embb_packet_loss')):8.6f} "
              f"({np.mean(get_recent('embb_packet_loss'))*100:.4f}%)")
        print(f"  Avg Power (W):      {np.mean(get_recent('embb_power')):8.2f} W")
        print(f"  PRB Efficiency:     {np.mean(get_recent('embb_prb_eff')):8.2f} Mbps/PRB")

        # === URLLC PERFORMANCE ===
        print(f"\n{'URLLC Performance (5 UEs)':-^80}")
        recent_lat = get_recent('urllc_latency')
        recent_lat_p95 = get_recent('urllc_latency_p95')
        recent_reliability = get_recent('urllc_reliability')

        lat = recent_lat  # raw latency samples

        print(f"  Latency (mean):     {np.mean(lat):8.4f} ms")
        print(f"  Latency (p95):      {np.percentile(lat, 95):8.4f} ms")
        print(f"  Latency (max):      {np.max(lat):8.4f} ms")
        print(f"  Queue Length:       {np.mean(get_recent('urllc_queue')):8.2f} Mbits")
        print(f"  Packet Loss:        {np.mean(get_recent('urllc_packet_loss')):8.6f} "
              f"({np.mean(get_recent('urllc_packet_loss'))*100:.4f}%)")
        print(f"  Reliability:        {np.mean(recent_reliability):8.6f} "
              f"({np.mean(recent_reliability)*100:.4f}%)")
        print(f"  Avg Power (W):      {np.mean(get_recent('urllc_power')):8.2f} W")
        print(f"  PRB Efficiency:     {np.mean(get_recent('urllc_prb_eff')):8.2f} Mbps/PRB")

        # === RESOURCE ALLOCATION ===
        print(f"\n{'Resource Allocation':-^80}")
        req_embb = get_recent_agent('prb_requested', 'embb')
        req_urllc = get_recent_agent('prb_requested', 'urllc')
        assign_embb = get_recent_agent('prb_assigned', 'embb')
        assign_urllc = get_recent_agent('prb_assigned', 'urllc')

        print(f"  eMBB  PRBs:")
        print(f"    Requested: {np.mean(req_embb):6.3f} \u00b1 {np.std(req_embb):5.3f}  "
              f"({np.mean(req_embb)*100:5.1f}%)")
        print(f"    Assigned:  {np.mean(assign_embb):6.3f} \u00b1 {np.std(assign_embb):5.3f}  "
              f"({np.mean(assign_embb)*100:5.1f}%)")

        print(f"  URLLC PRBs:")
        print(f"    Requested: {np.mean(req_urllc):6.3f} \u00b1 {np.std(req_urllc):5.3f}  "
              f"({np.mean(req_urllc)*100:5.1f}%)")
        print(f"    Assigned:  {np.mean(assign_urllc):6.3f} \u00b1 {np.std(assign_urllc):5.3f}  "
              f"({np.mean(assign_urllc)*100:5.1f}%)")

        total_assigned = np.mean([a + b for a, b in zip(assign_embb, assign_urllc)])
        print(f"  Total Assigned:    {total_assigned:6.3f}  ({total_assigned*100:5.1f}%)")

        print(f"{'='*80}\n")

    def plot_training(self, save_path: str = "training_results.png", smooth_window: int = 50):
        """Plot comprehensive training curves with all metrics"""

        def smooth(data, window):
            """Apply moving average smoothing"""
            if len(data) < window:
                return data
            smoothed = []
            for i in range(len(data)):
                start = max(0, i - window + 1)
                smoothed.append(np.mean(data[start:i+1]))
            return smoothed

        fig = plt.figure(figsize=(18, 16))
        gs = fig.add_gridspec(4, 3, hspace=0.5, wspace=0.4)

        # === ROW 1: Rewards & Constraints ===

        # Plot 1: Episode Rewards
        ax1 = fig.add_subplot(gs[0, 0])
        embb_rewards = list(self.metrics['episode_rewards']['embb'])
        urllc_rewards = list(self.metrics['episode_rewards']['urllc'])
        ax1.plot(embb_rewards, label='eMBB (raw)', alpha=0.2, color='C0')
        ax1.plot(smooth(embb_rewards, smooth_window), label=f'eMBB ({smooth_window}-ep)',
                alpha=0.8, linewidth=2, color='C0')
        ax1.plot(urllc_rewards, label='URLLC (raw)', alpha=0.2, color='C1')
        ax1.plot(smooth(urllc_rewards, smooth_window), label=f'URLLC ({smooth_window}-ep)',
                alpha=0.8, linewidth=2, color='C1')
        ax1.set_xlabel('Episode')
        ax1.set_ylabel('Reward')
        ax1.set_title('Episode Rewards')
        ax1.legend(fontsize=8)
        ax1.grid(True, alpha=0.3)

        # Plot 2: Costs and Lambda
        ax2 = fig.add_subplot(gs[0, 1])
        ax2_twin = ax2.twinx()
        costs_data = list(self.metrics['episode_costs'])
        lambda_data = list(self.metrics['lambda_dual'])
        ax2.plot(costs_data, 'b-', alpha=0.2)
        ax2.plot(smooth(costs_data, smooth_window), 'b-', label='Cost', alpha=0.8, linewidth=2)
        ax2_twin.plot(lambda_data, 'r-', label='Lambda', alpha=0.8, linewidth=2)
        ax2.axhline(y=0.2, color='g', linestyle='--', alpha=0.5, label='Cost Target')
        ax2.set_xlabel('Episode')
        ax2.set_ylabel('Cost', color='b')
        ax2_twin.set_ylabel('Lambda', color='r')
        ax2.set_title('Constraint Cost & Dual Variable')
        ax2.tick_params(axis='y', labelcolor='b')
        ax2_twin.tick_params(axis='y', labelcolor='r')
        ax2.legend(loc='upper left', fontsize=8)
        ax2_twin.legend(loc='upper right', fontsize=8)
        ax2.grid(True, alpha=0.3)

        # Plot 3: Violation Rate
        ax3 = fig.add_subplot(gs[0, 2])
        violations_data = list(self.metrics['constraint_violations'])
        ax3.plot(violations_data, alpha=0.2, color='C2')
        ax3.plot(smooth(violations_data, smooth_window), alpha=0.8, linewidth=2,
                color='C2', label=f'{smooth_window}-ep avg')
        ax3.axhline(y=20.0, color='orange', linestyle='--', label='Target (20%)', alpha=0.5)
        ax3.set_xlabel('Episode')
        ax3.set_ylabel('Violation Rate (%)')
        ax3.set_title('Constraint Violation Rate')
        ax3.legend(fontsize=8)
        ax3.grid(True, alpha=0.3)

        # === ROW 2: eMBB Metrics ===

        # Plot 4: eMBB Throughput
        ax4 = fig.add_subplot(gs[1, 0])
        throughput_data = list(self.metrics['embb_throughput'])
        ax4.plot(throughput_data, alpha=0.2, color='C0')
        ax4.plot(smooth(throughput_data, smooth_window), alpha=0.8, linewidth=2,
                color='C0', label=f'{smooth_window}-ep avg')
        ax4.axhline(y=50.0, color='g', linestyle='--', label='Min Req (50 Mbps)', alpha=0.5)
        ax4.set_xlabel('Episode')
        ax4.set_ylabel('Throughput (Mbps)')
        ax4.set_title('eMBB Throughput')
        ax4.legend(fontsize=8)
        ax4.grid(True, alpha=0.3)

        # Plot 5: eMBB Spectral Efficiency
        ax5 = fig.add_subplot(gs[1, 1])
        se_data = list(self.metrics['embb_spectral_eff'])
        ax5.plot(se_data, alpha=0.2, color='C3')
        ax5.plot(smooth(se_data, smooth_window), alpha=0.8, linewidth=2,
                color='C3', label=f'{smooth_window}-ep avg')
        ax5.set_xlabel('Episode')
        ax5.set_ylabel('Spectral Efficiency (bps/Hz)')
        ax5.set_title('eMBB Spectral Efficiency')
        ax5.legend(fontsize=8)
        ax5.grid(True, alpha=0.3)

        # Plot 6: eMBB PRB Efficiency
        ax6 = fig.add_subplot(gs[1, 2])
        prb_eff_data = list(self.metrics['embb_prb_eff'])
        ax6.plot(prb_eff_data, alpha=0.2, color='C4')
        ax6.plot(smooth(prb_eff_data, smooth_window), alpha=0.8, linewidth=2,
                color='C4', label=f'{smooth_window}-ep avg')
        ax6.set_xlabel('Episode')
        ax6.set_ylabel('PRB Efficiency (Mbps/PRB)')
        ax6.set_title('eMBB PRB Efficiency')
        ax6.legend(fontsize=8)
        ax6.grid(True, alpha=0.3)

        # === ROW 3: URLLC Metrics ===

        # Plot 7: URLLC Latency (with P95)
        ax7 = fig.add_subplot(gs[2, 0])
        latency_data = list(self.metrics['urllc_latency'])
        latency_p95_data = list(self.metrics['urllc_latency_p95'])
        ax7.plot(latency_data, alpha=0.2, color='C1')
        ax7.plot(smooth(latency_data, smooth_window), alpha=0.8, linewidth=2,
                color='C1', label=f'Mean ({smooth_window}-ep)')
        ax7.plot(smooth(latency_p95_data, smooth_window), alpha=0.8, linewidth=2,
                color='C5', linestyle='--', label=f'P95 ({smooth_window}-ep)')
        ax7.axhline(y=10.0, color='r', linestyle='--', label='Max (10ms)', alpha=0.5)
        ax7.axhline(y=1.0, color='orange', linestyle='--', label='Target (1ms)', alpha=0.5)
        ax7.set_xlabel('Episode')
        ax7.set_ylabel('Latency (ms)')
        ax7.set_title('URLLC Latency (Mean & P95)')
        ax7.legend(fontsize=8)
        ax7.grid(True, alpha=0.3)

        # Plot 8: URLLC Reliability
        ax8 = fig.add_subplot(gs[2, 1])
        reliability_data = [r * 100 for r in list(self.metrics['urllc_reliability'])]
        ax8.plot(reliability_data, alpha=0.2, color='C6')
        ax8.plot(smooth(reliability_data, smooth_window), alpha=0.8, linewidth=2,
                color='C6', label=f'{smooth_window}-ep avg')
        ax8.axhline(y=99.999, color='g', linestyle='--', label='Target (99.999%)', alpha=0.5)
        ax8.set_xlabel('Episode')
        ax8.set_ylabel('Reliability (%)')
        ax8.set_title('URLLC Reliability (1 - Packet Loss)')
        ax8.legend(fontsize=8)
        ax8.grid(True, alpha=0.3)

        # Plot 9: URLLC PRB Efficiency
        ax9 = fig.add_subplot(gs[2, 2])
        urllc_prb_eff_data = list(self.metrics['urllc_prb_eff'])
        ax9.plot(urllc_prb_eff_data, alpha=0.2, color='C7')
        ax9.plot(smooth(urllc_prb_eff_data, smooth_window), alpha=0.8, linewidth=2,
                color='C7', label=f'{smooth_window}-ep avg')
        ax9.set_xlabel('Episode')
        ax9.set_ylabel('PRB Efficiency (Mbps/PRB)')
        ax9.set_title('URLLC PRB Efficiency')
        ax9.legend(fontsize=8)
        ax9.grid(True, alpha=0.3)

        # === ROW 4: Resource Allocation ===

        # Plot 10: PRB Allocation
        ax10 = fig.add_subplot(gs[3, 0])
        req_embb = list(self.metrics['prb_requested']['embb'])
        req_urllc = list(self.metrics['prb_requested']['urllc'])
        assign_embb = list(self.metrics['prb_assigned']['embb'])
        assign_urllc = list(self.metrics['prb_assigned']['urllc'])

        ax10.plot(smooth(req_embb, smooth_window), label='eMBB Requested',
                 alpha=0.5, linestyle='--', color='C0', linewidth=1.5)
        ax10.plot(smooth(req_urllc, smooth_window), label='URLLC Requested',
                 alpha=0.5, linestyle='--', color='C1', linewidth=1.5)
        ax10.plot(smooth(assign_embb, smooth_window), label='eMBB Assigned',
                 alpha=0.8, linewidth=2, color='C0')
        ax10.plot(smooth(assign_urllc, smooth_window), label='URLLC Assigned',
                 alpha=0.8, linewidth=2, color='C1')

        total_assigned = [a + b for a, b in zip(assign_embb, assign_urllc)]
        ax10.plot(smooth(total_assigned, smooth_window), label='Total Assigned',
                 alpha=0.6, linestyle=':', linewidth=2.5, color='black')
        ax10.axhline(y=1.0, color='gray', linestyle='-', alpha=0.3)

        ax10.set_xlabel('Episode')
        ax10.set_ylabel('PRB Fraction')
        ax10.set_title(f'PRB Allocation ({smooth_window}-ep avg)')
        ax10.set_ylim([0, 1.2])
        ax10.legend(fontsize=8, loc='best')
        ax10.grid(True, alpha=0.3)

        # Plot 11: Power Levels
        ax11 = fig.add_subplot(gs[3, 1])
        embb_power = list(self.metrics['embb_power'])
        urllc_power = list(self.metrics['urllc_power'])
        ax11.plot(smooth(embb_power, smooth_window), label='eMBB Power',
                 alpha=0.8, linewidth=2, color='C0')
        ax11.plot(smooth(urllc_power, smooth_window), label='URLLC Power',
                 alpha=0.8, linewidth=2, color='C1')
        ax11.set_xlabel('Episode')
        ax11.set_ylabel('Power (Watts)')
        ax11.set_title(f'Transmission Power ({smooth_window}-ep avg)')
        ax11.set_ylim([0, 85])
        ax11.legend(fontsize=8)
        ax11.grid(True, alpha=0.3)

        # Plot 12: Packet Loss Comparison
        ax12 = fig.add_subplot(gs[3, 2])
        embb_loss = [l * 100 for l in list(self.metrics['embb_packet_loss'])]
        urllc_loss = [l * 100 for l in list(self.metrics['urllc_packet_loss'])]
        ax12.plot(smooth(embb_loss, smooth_window), label='eMBB',
                 alpha=0.8, linewidth=2, color='C0')
        ax12.plot(smooth(urllc_loss, smooth_window), label='URLLC',
                 alpha=0.8, linewidth=2, color='C1')
        ax12.set_xlabel('Episode')
        ax12.set_ylabel('Packet Loss (%)')
        ax12.set_title(f'Packet Loss ({smooth_window}-ep avg)')
        ax12.set_yscale('log')
        ax12.legend(fontsize=8)
        ax12.grid(True, alpha=0.3, which='both')

        fig.tight_layout(pad=3.0, h_pad=3.0, w_pad=2.0)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\n\u2713 Training plots saved to {save_path}")
        plt.show()
        plt.close()


def train_constrained_marl(
    num_episodes: int = 15000,
    max_steps: int = 288,
    lambda_lr: float = 0.002,
    lambda_max: float = 5.0,
    cost_limit: float = 0.15,
    eval_freq: int = 20,
    save_freq: int = 15000,
    save_dir: str = "./checkpoints",  # CHANGED: was a hardcoded Colab Drive path.
                                       # Pass your own path here, e.g.
                                       # "/content/drive/MyDrive/your_folder" on Colab.
):
    """
    Main training loop with constraint handling
    """
    # Ensure directory exists
    os.makedirs(save_dir, exist_ok=True)

    # Create environment
    env = ImprovedRANSlicingEnv(
        max_steps=max_steps,
        enable_mmtc=False,
        time_varying_traffic=True,
        enable_mobility=True,
        max_power_w=20.0,
        num_ues={"embb": 5, "urllc": 5},
    )

    # Get dimensions
    sample_obs, _ = env.reset()
    sample_state = env.state()

    obs_dim = sample_obs["embb_agent"].shape[0]
    state_dim = sample_state.shape[0]
    action_dim = 2

    print(f"\n{'='*70}")
    print(f"CONSTRAINED MAPPO TRAINING")
    print(f"{'='*70}")
    print(f"Observation dim: {obs_dim}, State dim: {state_dim}, Action dim: {action_dim}")
    print(f"Episodes: {num_episodes}, Max steps: {max_steps}")
    print(f"Cost limit: {cost_limit} (allows {cost_limit*100}% violations)")
    print(f"{'='*70}\n")

    # Create agents
    agents = {
        "embb_agent": MAPPO(obs_dim, state_dim, action_dim=2, hidden_dim=128, lr=3e-4),
        "urllc_agent": MAPPO(obs_dim, state_dim, action_dim=2, hidden_dim=128, lr=3e-4)
    }

    # Lagrangian multiplier
    lambda_dual = 0.0

    # Logger
    logger = TrainingLogger(window_size=50)

    # Training loop
    for ep in range(num_episodes):
        obs, _ = env.reset()

        # Episode memory
        memory = {
            a: {
                "obs": [], "states": [], "acts": [], "logps": [],
                "rews": [], "costs": [], "dones": []
            }
            for a in agents
        }

        episode_metrics = {
            # eMBB
            "embb_throughput": [],
            "embb_spectral_eff": [],
            "embb_packet_loss": [],
            "embb_power": [],
            "embb_prb_eff": [],
            "embb_queue": [],

            # URLLC
            "urllc_latency": [],
            "urllc_latency_p95": [],
            "urllc_packet_loss": [],
            "urllc_reliability": [],
            "urllc_power": [],
            "urllc_prb_eff": [],
            "urllc_queue": [],

            # Shared
            "urllc_costs": [],
            "embb_costs": [],
            "prb_requested_embb": [],
            "prb_requested_urllc": [],
            "prb_assigned_embb": [],
            "prb_assigned_urllc": [],
            "violation_magnitude": [],
            "urllc_violation_mag": [],
            "embb_violation_mag": [],
        }

        done = False

        # Rollout episode
        while not done:
            state = env.state()
            actions = {}
            logs = {}

            # Select actions
            for a in agents:
                act, logp = agents[a].select_action(obs[a])
                actions[a] = act
                logs[a] = logp

            # Environment step
            next_obs, rewards, terms, truncs, infos = env.step(actions)
            done = all(terms.values()) or all(truncs.values())

            # Share URLLC cost with both agents
            embb_cost = infos["embb_agent"]["cost"]
            urllc_cost = infos["urllc_agent"]["cost"]
            system_cost = 0.3 * embb_cost + 0.7 * urllc_cost

            # Store transitions
            for a in agents:
                memory[a]["obs"].append(obs[a])
                memory[a]["states"].append(state)
                memory[a]["acts"].append(actions[a])
                memory[a]["logps"].append(logs[a])
                memory[a]["rews"].append(rewards[a])
                memory[a]["costs"].append(system_cost)
                memory[a]["dones"].append(done)

            # Track metrics
            embb_m = infos["embb_agent"]["metrics"]
            urllc_m = infos["urllc_agent"]["metrics"]

            # --- URLLC ---
            lat = urllc_m["latency"]
            loss = urllc_m["packet_loss"]
            lat_v_urllc = min(1.0, max( 0.0, (lat - env.constraints["urllc"]["max_latency"]) / env.constraints["urllc"]["max_latency"] ))
            loss_v_urllc = min(1.0, max( 0.0, (loss - env.constraints["urllc"]["max_packet_loss"]) / env.constraints["urllc"]["max_packet_loss"] ))
            urllc_violation_mag = lat_v_urllc + loss_v_urllc

            # --- eMBB ---
            thr = embb_m["throughput"]
            lat_e = embb_m["latency"]
            thr_v_embb = min(1.0, max( 0.0, (env.constraints["embb"]["min_throughput"] - thr) / env.constraints["embb"]["min_throughput"] ))
            lat_v_embb = min(1.0, max( 0.0, (lat_e - env.constraints["embb"]["max_latency"]) / env.constraints["embb"]["max_latency"] ))
            embb_violation_mag = thr_v_embb + lat_v_embb

            # --- SYSTEM LEVEL ---
            system_violation_mag = 0.3 * embb_violation_mag + 0.7 * urllc_violation_mag
            episode_metrics["violation_magnitude"].append(system_violation_mag)
            episode_metrics["urllc_violation_mag"].append(urllc_violation_mag)
            episode_metrics["embb_violation_mag"].append(embb_violation_mag)

            # --- eMBB ---
            episode_metrics["embb_throughput"].append(embb_m["throughput"])
            episode_metrics["embb_spectral_eff"].append(embb_m["spectral_efficiency"])
            episode_metrics["embb_packet_loss"].append(embb_m["packet_loss"])
            episode_metrics["embb_queue"].append(embb_m["queue_length"])

            # --- URLLC ---
            episode_metrics["urllc_latency"].append(urllc_m["latency"])
            episode_metrics["urllc_packet_loss"].append(urllc_m["packet_loss"])
            episode_metrics["urllc_reliability"].append(1.0 - urllc_m["packet_loss"])
            episode_metrics["urllc_queue"].append(urllc_m["queue_length"])

            # --- Costs ---
            episode_metrics["urllc_costs"].append(urllc_cost)
            episode_metrics["embb_costs"].append(embb_cost)

            # --- PRBs ---
            episode_metrics["prb_requested_embb"].append(actions["embb_agent"][0])
            episode_metrics["prb_requested_urllc"].append(actions["urllc_agent"][0])

            prb_alloc = env.actual_prb_allocation
            assert abs( sum(prb_alloc.values()) - env.total_prbs ) < 1e-6, "PRB allocation mismatch!"
            episode_metrics["prb_assigned_embb"].append(prb_alloc["embb"] / env.total_prbs)
            episode_metrics["prb_assigned_urllc"].append(prb_alloc["urllc"] / env.total_prbs)

            # --- PRB efficiency ---
            episode_metrics["embb_prb_eff"].append(
                embb_m["throughput"] / max(prb_alloc["embb"], 1e-6)
            )
            episode_metrics["urllc_prb_eff"].append(
                urllc_m["throughput"] / max(prb_alloc["urllc"], 1e-6)
            )

            # --- Power ---
            power_alloc = env.actual_power_allocation
            assert sum(power_alloc.values()) <= env.max_power_w + 1e-3, (f"Power budget exceeded: {sum(power_alloc.values()):.2f}W > {env.max_power_w}W")
            episode_metrics["embb_power"].append(power_alloc["embb"])
            episode_metrics["urllc_power"].append(power_alloc["urllc"])

            obs = next_obs

        # Update policies
        for a in agents:
            data = (
                memory[a]["obs"],
                memory[a]["states"],
                memory[a]["acts"],
                memory[a]["logps"],
                memory[a]["rews"],
                memory[a]["costs"],
                memory[a]["dones"]
            )
            agents[a].update(data, lambda_dual)

        # Dual variable update (Lagrangian)
        avg_cost = np.mean(memory["urllc_agent"]["costs"])
        lambda_dual = np.clip(
            lambda_dual + lambda_lr * (avg_cost - cost_limit),
            0.0,
            lambda_max
        )

        latencies = episode_metrics["urllc_latency"]
        p95 = np.percentile(latencies, 95)
        lat_max = np.max(latencies)

        assert lat_max >= p95, "Latency statistics corrupted"

        # Log metrics
        urllc_violated = [c > 0 for c in episode_metrics["urllc_costs"]]
        embb_violated = [c > 0 for c in episode_metrics["embb_costs"]]
        system_violated = [u or e for u, e in zip(urllc_violated, embb_violated)]
        violation_rate = 100 * np.mean(system_violated)

        logger.log_episode(ep, {
            "episode_rewards": {
                "embb": sum(memory["embb_agent"]["rews"]),
                "urllc": sum(memory["urllc_agent"]["rews"])
            },
            "episode_costs": avg_cost,
            "lambda_dual": lambda_dual,

            # ==== Violations ====
            "constraint_violations": violation_rate,
            "violation_magnitude": np.mean(episode_metrics["violation_magnitude"]),
            "urllc_violation_magnitude": np.mean(episode_metrics["urllc_violation_mag"]),
            "embb_violation_magnitude": np.mean(episode_metrics["embb_violation_mag"]),

            # === eMBB metrics ===
            "embb_throughput": np.mean(episode_metrics["embb_throughput"]),
            "embb_spectral_eff": np.mean(episode_metrics["embb_spectral_eff"]),
            "embb_packet_loss": np.mean(episode_metrics["embb_packet_loss"]),
            "embb_power": np.mean(episode_metrics["embb_power"]),
            "embb_prb_eff": np.mean(episode_metrics["embb_prb_eff"]),
            "embb_queue": np.mean(episode_metrics["embb_queue"]),

            # === URLLC metrics ===
            "urllc_latency": np.mean(latencies),
            "urllc_latency_p95": p95,
            "urllc_latency_max": lat_max,
            "urllc_packet_loss": np.mean(episode_metrics["urllc_packet_loss"]),
            "urllc_reliability": np.mean(episode_metrics["urllc_reliability"]),
            "urllc_power": np.mean(episode_metrics["urllc_power"]),
            "urllc_prb_eff": np.mean(episode_metrics["urllc_prb_eff"]),
            "urllc_queue": np.mean(episode_metrics["urllc_queue"]),

            # === PRB stats ===
            "prb_requested": {
                "embb": np.mean(episode_metrics["prb_requested_embb"]),
                "urllc": np.mean(episode_metrics["prb_requested_urllc"])
            },
            "prb_assigned": {
                "embb": np.mean(episode_metrics["prb_assigned_embb"]),
                "urllc": np.mean(episode_metrics["prb_assigned_urllc"])
            }
        })

        # Print progress
        if (ep + 1) % eval_freq == 0:
            logger.print_summary(ep + 1)

        # Save checkpoint
        if (ep + 1) % save_freq == 0:
            checkpoint = {
                "episode": ep + 1,
                "agents": {a: agents[a].ac.state_dict() for a in agents},
                "lambda_dual": lambda_dual
            }
            # CHANGED: now saves into save_dir (previously wrote to the
            # current working directory, inconsistent with the final checkpoint)
            ckpt_path = os.path.join(save_dir, f"checkpoint_ep{ep+1}.pt")
            torch.save(checkpoint, ckpt_path)
            print(f"Checkpoint saved at episode {ep+1}")

        if (ep + 1) == num_episodes:
            ckpt_path = os.path.join(save_dir, f"checkpoint_ep{num_episodes}.pt")
            torch.save({
                "episode": num_episodes,
                "agents": {a: agents[a].ac.state_dict() for a in agents},
                "lambda_dual": lambda_dual
            }, ckpt_path)
            print(f"\n\u2713 Final checkpoint saved \u2192 {ckpt_path}")

            plot_path = os.path.join(save_dir, "final_training_results.png")
            logger.plot_training(plot_path)

    # Final plots
    logger.plot_training(os.path.join(save_dir, "final_training_results.png"))

    return agents, logger


if __name__ == "__main__":
    # Run training with FIXED parameters
    trained_agents, training_logger = train_constrained_marl(
        num_episodes=15000,
        max_steps=288,
        lambda_lr=0.002,
        lambda_max=5.0,
        cost_limit=0.15, #15%
        eval_freq=20,
        save_freq=5000,
        # CHANGED: was a hardcoded Colab Drive path. Set your own path here,
        # e.g. "/content/drive/MyDrive/your_folder" if running on Colab.
        save_dir="./checkpoints",
    )

    print("\n" + "="*70)
    print("Training completed successfully!")
    print("="*70)

## Static Slicing Baseline

This module provides **non-learning baseline strategies** for comparison against
the MAPPO agents. Instead of an RL policy, each slice uses a `StaticAgent` that
always returns the same fixed `[prb_fraction, power_fraction]` action — e.g. a
constant 50/50 split, or an eMBB-favoring 70/30 split.

Running these baselines through the *exact same* `ImprovedRANSlicingEnv`, reward
computation, cost computation, and metric-logging pipeline as the MAPPO training
script lets you directly answer: **"how much does learning actually help, versus
just picking a reasonable fixed allocation?"**

### Predefined strategies (`STRATEGIES`)

| Name | eMBB (PRB/Power) | URLLC (PRB/Power) | Use case |
|------|-------------------|---------------------|----------|
| `equal` | 50% / 50% | 50% / 50% | Neutral baseline |
| `embb_heavy` | 70% / 65% | 30% / 35% | Throughput-favoring |
| `urllc_heavy` | 30% / 35% | 70% / 65% | Latency-favoring |
| `embb_priority` | 80% / 75% | 20% / 25% | Strong eMBB priority |
| `urllc_priority` | 20% / 25% | 80% / 75% | Strong URLLC priority |
| `custom` | user-defined | user-defined | Any fixed split you specify |

### Key components

- **`StaticAgent`** — a drop-in replacement for the MAPPO agent's
  `select_action()` interface, but always returns the same fixed action
  (with a dummy log-probability of `0.0`, since no policy gradient is involved).
- **`TrainingLogger`** — same metric-tracking and plotting structure as the
  MAPPO training script, so training curves/plots are directly comparable side
  by side.
- **`worst_5_high` / `worst_5_low`** — tail-risk helpers that compute the mean
  of the worst 5% of samples, used for metrics where either high values are bad
  (latency) or low values are bad (throughput, reliability). This gives a sense
  of worst-case (not just average) performance.
- **`run_static_slicing(...)`** — the main evaluation loop: builds the static
  agents and environment, runs `num_episodes` full episodes while logging the
  same per-step and per-episode metrics as MAPPO, then prints both a windowed
  summary and final global step-level statistics (mean + worst-5% tail) across
  the entire run.

### Configuration

All run parameters are set in `__init__`/`run_static_slicing` arguments or in
the clearly-marked `CONFIGURATION` block under `if __name__ == "__main__":` — no
hardcoded personal paths. Output plots are saved to a relative filename
(`static_<strategy>_results.png`) in the working directory by default.

**To run a different strategy:** change the `STRATEGY` variable at the bottom of
the file to any key in `STRATEGIES`, or `"custom"` to use your own fractions via
`CUSTOM_SPLIT`.

In [ ]:
"""
Static Slicing Baseline
=======================
Runs fixed (static) PRB and power allocation strategies against the same
ImprovedRANSlicingEnv used by the MAPPO agents, collecting identical metrics
so results can be directly compared.

Static strategies available
---------------------------
  "equal"        - 50 / 50 split for both eMBB and URLLC
  "embb_heavy"   - 70 / 30 split favouring eMBB
  "urllc_heavy"  - 30 / 70 split favouring URLLC
  "custom"       - user-defined fractions via CLI / function argument
"""

import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import deque
from typing import Dict, List, Tuple


# -----------------------------------------------------------------------------
# Static "agent" - always returns the same action
# -----------------------------------------------------------------------------

class StaticAgent:
    """
    Mimics the MAPPO agent interface but always outputs a fixed action.

    The action vector for each agent is [prb_fraction, power_fraction],
    both in [0, 1], which is the same shape as the MAPPO actor output.
    """

    def __init__(self, prb_fraction: float, power_fraction: float):
        self.action = np.array([prb_fraction, power_fraction], dtype=np.float32)

    def select_action(self, obs: np.ndarray) -> Tuple[np.ndarray, float]:
        """Return fixed action with a dummy log-prob of 0.0."""
        return self.action.copy(), 0.0


# -----------------------------------------------------------------------------
# Logger - identical structure to TrainingLogger in the MAPPO script
# -----------------------------------------------------------------------------

class TrainingLogger:
    """Tracks and visualises evaluation metrics (mirrors MAPPO TrainingLogger)."""

    def __init__(self, window_size: int = 50):
        self.window_size = window_size
        self.metrics = {
            "episode_rewards":          {"embb": deque(maxlen=None), "urllc": deque(maxlen=None)},
            "episode_costs":            deque(maxlen=None),
            "constraint_violations":    deque(maxlen=None),
            "violation_magnitude":      deque(maxlen=None),
            "urllc_violation_magnitude": deque(maxlen=None),
            "embb_violation_magnitude": deque(maxlen=None),
            "embb_throughput":          deque(maxlen=None),
            "embb_spectral_eff":        deque(maxlen=None),
            "embb_packet_loss":         deque(maxlen=None),
            "embb_power":               deque(maxlen=None),
            "embb_prb_eff":             deque(maxlen=None),
            "urllc_latency":            deque(maxlen=None),
            "urllc_latency_p95":        deque(maxlen=None),
            "urllc_latency_max":        deque(maxlen=None),
            "urllc_packet_loss":        deque(maxlen=None),
            "urllc_reliability":        deque(maxlen=None),
            "urllc_power":              deque(maxlen=None),
            "urllc_prb_eff":            deque(maxlen=None),
            "prb_requested":            {"embb": deque(maxlen=None), "urllc": deque(maxlen=None)},
            "prb_assigned":             {"embb": deque(maxlen=None), "urllc": deque(maxlen=None)},
        }

    def log_episode(self, ep: int, metrics: dict):
        for key, value in metrics.items():
            if key in ("episode_rewards", "prb_requested", "prb_assigned"):
                for agent, val in value.items():
                    self.metrics[key][agent].append(val)
            else:
                self.metrics[key].append(value)

    def _get_recent(self, key):
        return list(self.metrics[key])[-self.window_size:]

    def _get_recent_agent(self, key, agent):
        return list(self.metrics[key][agent])[-self.window_size:]

    def print_summary(self, ep: int, label: str = "Static"):
        print(f"\n{'='*80}")
        print(f"[{label}]  Episode {ep:4d} Summary")
        print(f"{'='*80}")

        if not self.metrics["episode_rewards"]["embb"]:
            print("No data yet.")
            return

        recent_embb_r  = self._get_recent_agent("episode_rewards", "embb")
        recent_urllc_r = self._get_recent_agent("episode_rewards", "urllc")
        recent_costs   = self._get_recent("episode_costs")
        recent_viol    = self._get_recent("constraint_violations")

        print(f"\n{'Rewards & Constraints':-^80}")
        print(f"  Episode Rewards (last {self.window_size}):")
        print(f"    eMBB:  {np.mean(recent_embb_r):8.2f} \u00b1 {np.std(recent_embb_r):6.2f}")
        print(f"    URLLC: {np.mean(recent_urllc_r):8.2f} \u00b1 {np.std(recent_urllc_r):6.2f}")
        print(f"  Constraint Cost:  {np.mean(recent_costs):8.4f} \u00b1 {np.std(recent_costs):6.4f}")

        print(f"\n{'Violations':-^80}")
        print(f"  Violation Frequency:           {np.mean(recent_viol):8.2f}%")
        print(f"  Violation Magnitude (System):  {100 * np.mean(self._get_recent('violation_magnitude')):8.4f}%")
        print(f"  Violation Magnitude (URLLC):   {100 * np.mean(self._get_recent('urllc_violation_magnitude')):8.4f}%")
        print(f"  Violation Magnitude (eMBB):    {100 * np.mean(self._get_recent('embb_violation_magnitude')):8.4f}%")

        print(f"\n{'eMBB Performance':-^80}")
        print(f"  Throughput:         {np.mean(self._get_recent('embb_throughput')):8.2f} Mbps  (target: \u226550 Mbps)")
        print(f"  Spectral Eff:       {np.mean(self._get_recent('embb_spectral_eff')):8.2f} bps/Hz")
        print(f"  Packet Loss:        {np.mean(self._get_recent('embb_packet_loss')):8.6f}  "
              f"({np.mean(self._get_recent('embb_packet_loss'))*100:.4f}%)")
        print(f"  Avg Power (W):      {np.mean(self._get_recent('embb_power')):8.2f} W")
        print(f"  PRB Efficiency:     {np.mean(self._get_recent('embb_prb_eff')):8.2f} Mbps/PRB")

        lat = self._get_recent("urllc_latency")
        print(f"\n{'URLLC Performance':-^80}")
        print(f"  Latency (mean):     {np.mean(lat):8.4f} ms")
        print(f"  Latency (p95):      {np.percentile(lat, 95):8.4f} ms")
        print(f"  Latency (max):      {np.max(lat):8.4f} ms")
        print(f"  Packet Loss:        {np.mean(self._get_recent('urllc_packet_loss')):8.6f}  "
              f"({np.mean(self._get_recent('urllc_packet_loss'))*100:.4f}%)")
        print(f"  Reliability:        {np.mean(self._get_recent('urllc_reliability')):8.6f}  "
              f"({np.mean(self._get_recent('urllc_reliability'))*100:.4f}%)")
        print(f"  Avg Power (W):      {np.mean(self._get_recent('urllc_power')):8.2f} W")
        print(f"  PRB Efficiency:     {np.mean(self._get_recent('urllc_prb_eff')):8.2f} Mbps/PRB")

        req_embb   = self._get_recent_agent("prb_requested", "embb")
        req_urllc  = self._get_recent_agent("prb_requested", "urllc")
        asgn_embb  = self._get_recent_agent("prb_assigned",  "embb")
        asgn_urllc = self._get_recent_agent("prb_assigned",  "urllc")
        total      = np.mean([a + b for a, b in zip(asgn_embb, asgn_urllc)])

        print(f"\n{'Resource Allocation':-^80}")
        print(f"  eMBB  PRBs: req={np.mean(req_embb):.3f}  assigned={np.mean(asgn_embb):.3f}")
        print(f"  URLLC PRBs: req={np.mean(req_urllc):.3f}  assigned={np.mean(asgn_urllc):.3f}")
        print(f"  Total Assigned:    {total:.3f}  ({total*100:.1f}%)")
        print(f"{'='*80}\n")

    # NOTE: a broken/unused duplicate of the module-level worst_5_* helpers
    # (missing `self`, never called) was removed here during cleanup.

    def plot_training(
        self,
        save_path: str = "static_slicing_results.png",
        smooth_window: int = 50,
        label: str = "Static Slicing",
    ):
        def smooth(data, w):
            if len(data) < w:
                return data
            out = []
            for i in range(len(data)):
                s = max(0, i - w + 1)
                out.append(np.mean(data[s:i+1]))
            return out

        fig = plt.figure(figsize=(18, 16))
        gs = fig.add_gridspec(4, 3, hspace=0.5, wspace=0.4)
        fig.suptitle(f"Static Slicing Baseline \u2013 {label}", fontsize=14, fontweight="bold")

        # -- Row 1: Rewards & Constraints ----------------------------------
        ax1 = fig.add_subplot(gs[0, 0])
        embb_r  = list(self.metrics["episode_rewards"]["embb"])
        urllc_r = list(self.metrics["episode_rewards"]["urllc"])
        ax1.plot(embb_r,  alpha=0.2, color="C0")
        ax1.plot(smooth(embb_r,  smooth_window), alpha=0.8, lw=2, color="C0", label=f"eMBB ({smooth_window}-ep)")
        ax1.plot(urllc_r, alpha=0.2, color="C1")
        ax1.plot(smooth(urllc_r, smooth_window), alpha=0.8, lw=2, color="C1", label=f"URLLC ({smooth_window}-ep)")
        ax1.set(xlabel="Episode", ylabel="Reward", title="Episode Rewards")
        ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

        ax2 = fig.add_subplot(gs[0, 1])
        costs_data = list(self.metrics["episode_costs"])
        ax2.plot(costs_data, alpha=0.2, color="b")
        ax2.plot(smooth(costs_data, smooth_window), "b-", lw=2, label="Cost")
        ax2.axhline(y=0.2, color="g", ls="--", alpha=0.5, label="Cost Target")
        ax2.set(xlabel="Episode", ylabel="Cost", title="Constraint Cost")
        ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

        ax3 = fig.add_subplot(gs[0, 2])
        viol_data = list(self.metrics["constraint_violations"])
        ax3.plot(viol_data, alpha=0.2, color="C2")
        ax3.plot(smooth(viol_data, smooth_window), alpha=0.8, lw=2, color="C2",
                 label=f"{smooth_window}-ep avg")
        ax3.axhline(y=20.0, color="orange", ls="--", alpha=0.5, label="Target (20%)")
        ax3.set(xlabel="Episode", ylabel="Violation Rate (%)", title="Constraint Violation Rate")
        ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

        # -- Row 2: eMBB ------------------------------------------------------
        ax4 = fig.add_subplot(gs[1, 0])
        thr_data = list(self.metrics["embb_throughput"])
        ax4.plot(thr_data, alpha=0.2, color="C0")
        ax4.plot(smooth(thr_data, smooth_window), alpha=0.8, lw=2, color="C0",
                 label=f"{smooth_window}-ep avg")
        ax4.axhline(y=50.0, color="g", ls="--", alpha=0.5, label="Min (50 Mbps)")
        ax4.set(xlabel="Episode", ylabel="Throughput (Mbps)", title="eMBB Throughput")
        ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

        ax5 = fig.add_subplot(gs[1, 1])
        se_data = list(self.metrics["embb_spectral_eff"])
        ax5.plot(se_data, alpha=0.2, color="C3")
        ax5.plot(smooth(se_data, smooth_window), alpha=0.8, lw=2, color="C3",
                 label=f"{smooth_window}-ep avg")
        ax5.set(xlabel="Episode", ylabel="Spectral Eff (bps/Hz)", title="eMBB Spectral Efficiency")
        ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)

        ax6 = fig.add_subplot(gs[1, 2])
        prb_eff_data = list(self.metrics["embb_prb_eff"])
        ax6.plot(prb_eff_data, alpha=0.2, color="C4")
        ax6.plot(smooth(prb_eff_data, smooth_window), alpha=0.8, lw=2, color="C4",
                 label=f"{smooth_window}-ep avg")
        ax6.set(xlabel="Episode", ylabel="PRB Eff (Mbps/PRB)", title="eMBB PRB Efficiency")
        ax6.legend(fontsize=8); ax6.grid(True, alpha=0.3)

        # -- Row 3: URLLC ------------------------------------------------------
        ax7 = fig.add_subplot(gs[2, 0])
        lat_data   = list(self.metrics["urllc_latency"])
        lat_p95    = list(self.metrics["urllc_latency_p95"])
        ax7.plot(lat_data, alpha=0.2, color="C1")
        ax7.plot(smooth(lat_data, smooth_window), alpha=0.8, lw=2, color="C1",
                 label=f"Mean ({smooth_window}-ep)")
        ax7.plot(smooth(lat_p95, smooth_window), alpha=0.8, lw=2, color="C5", ls="--",
                 label=f"P95 ({smooth_window}-ep)")
        ax7.axhline(y=10.0, color="r", ls="--", alpha=0.5, label="Max (10ms)")
        ax7.axhline(y=1.0, color="orange", ls="--", alpha=0.5, label="Target (1ms)")
        ax7.set(xlabel="Episode", ylabel="Latency (ms)", title="URLLC Latency (Mean & P95)",
                ylim=[0, 20])
        ax7.legend(fontsize=8); ax7.grid(True, alpha=0.3)

        ax8 = fig.add_subplot(gs[2, 1])
        rel_data = [r * 100 for r in list(self.metrics["urllc_reliability"])]
        ax8.plot(rel_data, alpha=0.2, color="C6")
        ax8.plot(smooth(rel_data, smooth_window), alpha=0.8, lw=2, color="C6",
                 label=f"{smooth_window}-ep avg")
        ax8.axhline(y=99.999, color="g", ls="--", alpha=0.5, label="Target (99.999%)")
        ax8.set(xlabel="Episode", ylabel="Reliability (%)", title="URLLC Reliability",
                ylim=[99.9, 100.0])
        ax8.legend(fontsize=8); ax8.grid(True, alpha=0.3)

        ax9 = fig.add_subplot(gs[2, 2])
        u_prb_eff = list(self.metrics["urllc_prb_eff"])
        ax9.plot(u_prb_eff, alpha=0.2, color="C7")
        ax9.plot(smooth(u_prb_eff, smooth_window), alpha=0.8, lw=2, color="C7",
                 label=f"{smooth_window}-ep avg")
        ax9.set(xlabel="Episode", ylabel="PRB Eff (Mbps/PRB)", title="URLLC PRB Efficiency")
        ax9.legend(fontsize=8); ax9.grid(True, alpha=0.3)

        # -- Row 4: Resources ----------------------------------------------
        ax10 = fig.add_subplot(gs[3, 0])
        req_e  = list(self.metrics["prb_requested"]["embb"])
        req_u  = list(self.metrics["prb_requested"]["urllc"])
        asgn_e = list(self.metrics["prb_assigned"]["embb"])
        asgn_u = list(self.metrics["prb_assigned"]["urllc"])
        total  = [a + b for a, b in zip(asgn_e, asgn_u)]
        ax10.plot(smooth(req_e,  smooth_window), ls="--", alpha=0.5, lw=1.5, color="C0", label="eMBB Req")
        ax10.plot(smooth(req_u,  smooth_window), ls="--", alpha=0.5, lw=1.5, color="C1", label="URLLC Req")
        ax10.plot(smooth(asgn_e, smooth_window), alpha=0.8, lw=2,   color="C0", label="eMBB Assigned")
        ax10.plot(smooth(asgn_u, smooth_window), alpha=0.8, lw=2,   color="C1", label="URLLC Assigned")
        ax10.plot(smooth(total,  smooth_window), ls=":", lw=2.5, color="black", alpha=0.6, label="Total")
        ax10.axhline(y=1.0, color="gray", ls="-", alpha=0.3)
        ax10.set(xlabel="Episode", ylabel="PRB Fraction", title=f"PRB Allocation ({smooth_window}-ep avg)",
                 ylim=[0, 1.2])
        ax10.legend(fontsize=8); ax10.grid(True, alpha=0.3)

        ax11 = fig.add_subplot(gs[3, 1])
        ep_pw = list(self.metrics["embb_power"])
        up_pw = list(self.metrics["urllc_power"])
        ax11.plot(smooth(ep_pw, smooth_window), alpha=0.8, lw=2, color="C0", label="eMBB")
        ax11.plot(smooth(up_pw, smooth_window), alpha=0.8, lw=2, color="C1", label="URLLC")
        ax11.set(xlabel="Episode", ylabel="Power (W)", title=f"Transmission Power ({smooth_window}-ep avg)",
                 ylim=[0, 85])
        ax11.legend(fontsize=8); ax11.grid(True, alpha=0.3)

        ax12 = fig.add_subplot(gs[3, 2])
        e_loss = [l * 100 for l in list(self.metrics["embb_packet_loss"])]
        u_loss = [l * 100 for l in list(self.metrics["urllc_packet_loss"])]
        ax12.plot(smooth(e_loss, smooth_window), alpha=0.8, lw=2, color="C0", label="eMBB")
        ax12.plot(smooth(u_loss, smooth_window), alpha=0.8, lw=2, color="C1", label="URLLC")
        ax12.set_yscale("log")
        ax12.set(xlabel="Episode", ylabel="Packet Loss (%)", title=f"Packet Loss ({smooth_window}-ep avg)")
        ax12.legend(fontsize=8); ax12.grid(True, alpha=0.3, which="both")

        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"\n\u2713 Plots saved to {save_path}")
        plt.close()

def worst_5_high(data):
    """Worst 5% for metrics where HIGH is bad (latency)"""
    sorted_data = sorted(data)
    n = len(sorted_data)
    cutoff = int(0.95 * n)
    worst = sorted_data[cutoff:]
    return sum(worst) / len(worst)

def worst_5_low(data):
    """Worst 5% for metrics where LOW is bad (throughput, reliability)"""
    sorted_data = sorted(data)
    n = len(sorted_data)
    cutoff = max(1, int(0.05 * n))
    worst = sorted_data[:cutoff]
    return sum(worst) / len(worst)


# -----------------------------------------------------------------------------
# Predefined static strategies
# -----------------------------------------------------------------------------

STRATEGIES: Dict[str, Dict[str, Tuple[float, float]]] = {
    # strategy_name -> {agent_name: (prb_fraction, power_fraction)}
    "equal": {
        "embb_agent":  (0.50, 0.50),
        "urllc_agent": (0.50, 0.50),
    },
    "embb_heavy": {
        "embb_agent":  (0.70, 0.65),
        "urllc_agent": (0.30, 0.35),
    },
    "urllc_heavy": {
        "embb_agent":  (0.30, 0.35),
        "urllc_agent": (0.70, 0.65),
    },
    "urllc_priority": {
        "embb_agent":  (0.20, 0.25),
        "urllc_agent": (0.80, 0.75),
    },
    "embb_priority": {
        "embb_agent":  (0.80, 0.75),
        "urllc_agent": (0.20, 0.25),
    },
}


# -----------------------------------------------------------------------------
# Main evaluation loop
# -----------------------------------------------------------------------------

def run_static_slicing(
    strategy: str = "equal",
    num_episodes: int = 500,
    max_steps: int = 100,
    cost_limit: float = 0.15,
    eval_freq: int = 20,
    # custom fractions (only used when strategy == "custom")
    embb_prb: float = 0.50,
    embb_pwr: float = 0.50,
    urllc_prb: float = 0.50,
    urllc_pwr: float = 0.50,
    save_plot: str = "static_slicing_results.png",
):
    """
    Evaluate static (fixed) PRB+power allocation over *num_episodes* episodes
    using the same ImprovedRANSlicingEnv and metric collection as MAPPO.
    """

    # -- Build static agents --------------------------------------------------
    if strategy == "custom":
        agent_cfg = {
            "embb_agent":  (embb_prb,  embb_pwr),
            "urllc_agent": (urllc_prb, urllc_pwr),
        }
    elif strategy in STRATEGIES:
        agent_cfg = STRATEGIES[strategy]
    else:
        raise ValueError(
            f"Unknown strategy '{strategy}'. "
            f"Choose from {list(STRATEGIES.keys()) + ['custom']}."
        )

    agents = {name: StaticAgent(*cfg) for name, cfg in agent_cfg.items()}

    label = (
        strategy if strategy != "custom"
        else f"custom(eMBB={embb_prb:.2f}/{embb_pwr:.2f}, URLLC={urllc_prb:.2f}/{urllc_pwr:.2f})"
    )

    # -- Env --------------------------------------------------------------------
    env = ImprovedRANSlicingEnv(
        max_steps=max_steps,
        enable_mmtc=False,
        time_varying_traffic=True,
        enable_mobility=True,
        max_power_w=20.0,
        num_ues={"embb": 5, "urllc": 5},
    )

    print(f"\n{'='*70}")
    print(f"STATIC SLICING BASELINE  \u2013  strategy: {label}")
    print(f"{'='*70}")
    print(f"Episodes: {num_episodes},  Max steps: {max_steps}")
    print(f"Cost limit: {cost_limit} ({cost_limit*100:.0f}% violations allowed)")
    print(f"Agent actions (fixed):")
    for name, (prb, pwr) in agent_cfg.items():
        print(f"  {name}: prb_frac={prb:.2f},  pwr_frac={pwr:.2f}")
    print(f"{'='*70}\n")

    logger = TrainingLogger(window_size=50)

    all_latencies_urllc = []
    all_throughput_embb = []
    all_reliability_urllc = []

    all_latency_embb = []
    all_throughput_urllc = []
    all_reliability_embb = []

    # -- Episode loop -------------------------------------------------------------
    for ep in range(num_episodes):
        obs, _ = env.reset()

        # Accumulators (same keys as MAPPO training loop)
        ep_rews  = {"embb_agent": 0.0, "urllc_agent": 0.0}
        ep_costs: List[float] = []

        ep_metrics = {
            "embb_throughput":    [],
            "embb_spectral_eff":  [],
            "embb_packet_loss":   [],
            "embb_power":         [],
            "embb_prb_eff":       [],
            "urllc_latency":      [],
            "urllc_latency_p95":  [],
            "urllc_packet_loss":  [],
            "urllc_reliability":  [],
            "urllc_power":        [],
            "urllc_prb_eff":      [],
            "urllc_costs":        [],
            "embb_costs":         [],
            "prb_requested_embb": [],
            "prb_requested_urllc":[],
            "prb_assigned_embb":  [],
            "prb_assigned_urllc": [],
            "violation_magnitude":     [],
            "urllc_violation_mag":     [],
            "embb_violation_mag":      [],
        }

        done = False

        while not done:
            actions = {}
            for a in agents:
                act, _ = agents[a].select_action(obs[a])
                actions[a] = act

            next_obs, rewards, terms, truncs, infos = env.step(actions)
            done = all(terms.values())

            # -- Costs (identical weighting as MAPPO) ---------------------------
            embb_cost  = infos["embb_agent"]["cost"]
            urllc_cost = infos["urllc_agent"]["cost"]
            system_cost = 0.3 * embb_cost + 0.7 * urllc_cost

            ep_costs.append(system_cost)
            for a in agents:
                ep_rews[a] += rewards[a]

            # -- Per-step metrics -------------------------------------------------
            embb_m  = infos["embb_agent"]["metrics"]
            urllc_m = infos["urllc_agent"]["metrics"]

            all_latencies_urllc.append(urllc_m["latency"])
            all_throughput_embb.append(embb_m["throughput"])
            all_reliability_urllc.append(1.0 - urllc_m["packet_loss"])

            all_latency_embb.append(embb_m["latency"])
            all_throughput_urllc.append(urllc_m["throughput"])
            all_reliability_embb.append(1.0 - embb_m["packet_loss"])

            # URLLC violation magnitudes
            lat      = urllc_m["latency"]
            loss_u   = urllc_m["packet_loss"]
            lat_v_u  = min(1.0, max(0.0, (lat  - env.constraints["urllc"]["max_latency"])   / env.constraints["urllc"]["max_latency"]))
            loss_v_u = min(1.0, max(0.0, (loss_u - env.constraints["urllc"]["max_packet_loss"]) / env.constraints["urllc"]["max_packet_loss"]))
            urllc_viol_mag = lat_v_u + loss_v_u

            # eMBB violation magnitudes
            thr    = embb_m["throughput"]
            lat_e  = embb_m["latency"]
            thr_v  = min(1.0, max(0.0, (env.constraints["embb"]["min_throughput"] - thr)  / env.constraints["embb"]["min_throughput"]))
            lat_v_e = min(1.0, max(0.0, (lat_e - env.constraints["embb"]["max_latency"])  / env.constraints["embb"]["max_latency"]))
            embb_viol_mag = thr_v + lat_v_e

            sys_viol_mag = 0.3 * embb_viol_mag + 0.7 * urllc_viol_mag

            ep_metrics["violation_magnitude"].append(sys_viol_mag)
            ep_metrics["urllc_violation_mag"].append(urllc_viol_mag)
            ep_metrics["embb_violation_mag"].append(embb_viol_mag)

            ep_metrics["embb_throughput"].append(embb_m["throughput"])
            ep_metrics["embb_spectral_eff"].append(embb_m["spectral_efficiency"])
            ep_metrics["embb_packet_loss"].append(embb_m["packet_loss"])

            ep_metrics["urllc_latency"].append(urllc_m["latency"])
            ep_metrics["urllc_packet_loss"].append(urllc_m["packet_loss"])
            ep_metrics["urllc_reliability"].append(1.0 - urllc_m["packet_loss"])

            ep_metrics["urllc_costs"].append(urllc_cost)
            ep_metrics["embb_costs"].append(embb_cost)

            ep_metrics["prb_requested_embb"].append(actions["embb_agent"][0])
            ep_metrics["prb_requested_urllc"].append(actions["urllc_agent"][0])

            prb_alloc = env.actual_prb_allocation
            assert abs(prb_alloc["embb"] + prb_alloc["urllc"] - env.total_prbs) < 1e-6, \
                "PRB allocation mismatch!"
            ep_metrics["prb_assigned_embb"].append(prb_alloc["embb"] / env.total_prbs)
            ep_metrics["prb_assigned_urllc"].append(prb_alloc["urllc"] / env.total_prbs)

            ep_metrics["embb_prb_eff"].append(
                embb_m["throughput"] / max(prb_alloc["embb"], 1e-6)
            )
            ep_metrics["urllc_prb_eff"].append(
                urllc_m["throughput"] / max(prb_alloc["urllc"], 1e-6)
            )

            power_alloc = env.actual_power_allocation
            assert sum(power_alloc.values()) <= env.max_power_w + 1e-3, \
                f"Power budget exceeded: {sum(power_alloc.values()):.2f}W > {env.max_power_w}W"
            ep_metrics["embb_power"].append(power_alloc["embb"])
            ep_metrics["urllc_power"].append(power_alloc["urllc"])

            obs = next_obs

        # -- Episode-level logging ---------------------------------------------
        avg_cost     = float(np.mean(ep_costs))
        latencies    = ep_metrics["urllc_latency"]
        p95          = float(worst_5_high(latencies))
        lat_max      = float(np.max(latencies))

        assert lat_max >= p95, "Latency statistics corrupted"

        urllc_violated = [c > 0 for c in ep_metrics["urllc_costs"]]
        embb_violated  = [c > 0 for c in ep_metrics["embb_costs"]]
        sys_violated   = [u or e for u, e in zip(urllc_violated, embb_violated)]
        violation_rate = 100.0 * float(np.mean(sys_violated))

        logger.log_episode(ep, {
            "episode_rewards": {
                "embb":  ep_rews["embb_agent"],
                "urllc": ep_rews["urllc_agent"],
            },
            "episode_costs":    avg_cost,

            "constraint_violations":     violation_rate,
            "violation_magnitude":       float(np.mean(ep_metrics["violation_magnitude"])),
            "urllc_violation_magnitude": float(np.mean(ep_metrics["urllc_violation_mag"])),
            "embb_violation_magnitude":  float(np.mean(ep_metrics["embb_violation_mag"])),

            "embb_throughput":   float(np.mean(ep_metrics["embb_throughput"])),
            "embb_spectral_eff": float(np.mean(ep_metrics["embb_spectral_eff"])),
            "embb_packet_loss":  float(np.mean(ep_metrics["embb_packet_loss"])),
            "embb_power":        float(np.mean(ep_metrics["embb_power"])),
            "embb_prb_eff":      float(np.mean(ep_metrics["embb_prb_eff"])),

            "urllc_latency":     float(np.mean(latencies)),
            "urllc_latency_p95": p95,
            "urllc_latency_max": lat_max,
            "urllc_packet_loss": float(np.mean(ep_metrics["urllc_packet_loss"])),
            "urllc_reliability": float(np.mean(ep_metrics["urllc_reliability"])),
            "urllc_power":       float(np.mean(ep_metrics["urllc_power"])),
            "urllc_prb_eff":     float(np.mean(ep_metrics["urllc_prb_eff"])),

            "prb_requested": {
                "embb":  float(np.mean(ep_metrics["prb_requested_embb"])),
                "urllc": float(np.mean(ep_metrics["prb_requested_urllc"])),
            },
            "prb_assigned": {
                "embb":  float(np.mean(ep_metrics["prb_assigned_embb"])),
                "urllc": float(np.mean(ep_metrics["prb_assigned_urllc"])),
            },
        })

        if (ep + 1) % eval_freq == 0:
            logger.print_summary(ep + 1, label=label)

    # -- Final plot ----------------------------------------------------------------
    logger.plot_training(save_path=save_plot, label=label)

    # ============================================================
    # FINAL GLOBAL STEP-LEVEL STATISTICS (ALL STEPS)
    # ============================================================
    print("\n" + "=" * 75)
    print("STATIC SLICING - GLOBAL STEP-LEVEL METRICS (ALL STEPS)")
    print("=" * 75)

    print("\nURLLC:")
    print(f"Latency Avg:        {np.mean(all_latencies_urllc):8.4f} ms")
    print(f"Latency 95th:       {worst_5_high(all_latencies_urllc):8.4f} ms")
    print(f"Throughput Avg:     {np.mean(all_throughput_urllc):8.4f} Mbps")
    print(f"Throughput 95th:    {worst_5_low(all_throughput_urllc):8.4f} Mbps")
    print(f"Reliability Avg:    {np.mean(all_reliability_urllc)*100:8.5f} %")
    print(f"Reliability 95th:   {worst_5_low(all_reliability_urllc)*100:8.5f} %")

    print("\neMBB:")
    print(f"Latency Avg:        {np.mean(all_latency_embb):8.4f} ms")
    print(f"Latency 95th:       {worst_5_high(all_latency_embb):8.4f} ms")
    print(f"Throughput Avg:     {np.mean(all_throughput_embb):8.4f} Mbps")
    print(f"Throughput 95th:    {worst_5_low(all_throughput_embb):8.4f} Mbps")
    print(f"Reliability Avg:    {np.mean(all_reliability_embb)*100:8.5f} %")
    print(f"Reliability 95th:   {worst_5_low(all_reliability_embb)*100:8.5f} %")

    print("=" * 75)

    return logger


if __name__ == "__main__":
    # =========================================================================
    # CONFIGURATION  -  edit this block to change the run
    # =========================================================================

    # -- Training duration -------------------------------------------------------
    NUM_EPISODES = 7500
    MAX_STEPS    = 100
    COST_LIMIT   = 0.15   # 15 % violation budget (same as MAPPO run)
    EVAL_FREQ    = 20     # print summary every N episodes

    # -- Strategy selection --------------------------------------------------------
    # Set STRATEGY to one of the keys below, or to "custom" to use
    # the fractions you define in the CUSTOM SPLIT section.
    #
    #   "equal"          -  50 / 50  PRB & power
    #   "embb_heavy"     -  70 / 30  favouring eMBB
    #   "urllc_heavy"    -  30 / 70  favouring URLLC
    #   "embb_priority"  -  20 / 80  strong eMBB priority
    #   "urllc_priority" -  20 / 80  strong URLLC priority
    #   "custom"         -  uses the custom fractions defined below
    #

    STRATEGY = "embb_priority"

    # -- Custom split (only used when STRATEGY = "custom") ------------------------
    # Each fraction is in [0.0, 1.0].  The environment handles the actual
    # PRB/power arbitration, so these are the *requested* fractions passed as
    # the agent action, exactly as the MAPPO actor would output them.
    CUSTOM_SPLIT = {
        "embb_agent":  {"prb": 0.50, "power": 0.50},
        "urllc_agent": {"prb": 0.50, "power": 0.50},
    }

    # =========================================================================
    # END OF CONFIGURATION  -  no need to edit below this line
    # =========================================================================

    if STRATEGY == "custom":
        run_static_slicing(
            strategy="custom",
            num_episodes=NUM_EPISODES,
            max_steps=MAX_STEPS,
            cost_limit=COST_LIMIT,
            eval_freq=EVAL_FREQ,
            embb_prb=CUSTOM_SPLIT["embb_agent"]["prb"],
            embb_pwr=CUSTOM_SPLIT["embb_agent"]["power"],
            urllc_prb=CUSTOM_SPLIT["urllc_agent"]["prb"],
            urllc_pwr=CUSTOM_SPLIT["urllc_agent"]["power"],
            save_plot="static_custom_results.png",
        )
    else:
        run_static_slicing(
            strategy=STRATEGY,
            num_episodes=NUM_EPISODES,
            max_steps=MAX_STEPS,
            cost_limit=COST_LIMIT,
            eval_freq=EVAL_FREQ,
            save_plot=f"static_{STRATEGY}_results.png",
        )

    print("\n" + "=" * 70)
    print("Static slicing evaluation completed.")
    print("=" * 70)